# **Time to Purchase & Cross-Session Product Exploration and Consideration**




## **Comprehensive Customer Journey Analysis**

This notebook provides an in-depth analysis of customer behavior within the Versace e-commerce dataset (`IMA2026_Versace_Time_To_Purchase.csv`), focusing on key aspects of the purchase journey. The comprehensive approach includes:

*   **Data Preparation & Exploratory Analysis**: Initial data loading, rigorous cleaning (missing values, outliers, duplicates), and extensive EDA to understand overall dataset characteristics, user engagement, and early sales trends.
*   **Time-to-Purchase (TTP) Framework**: Construction of a specialized dataset (`ttp_df`) at the user-item level to model the duration from first view to purchase, incorporating key behavioral features.
*   **UTM and Campaign Attribution Analysis**: Investigation of traffic sources and marketing campaigns to understand their impact on user journeys and conversion timing.
*   **Behavioral Customer Segmentation**: Application of K-means clustering to identify distinct user personas based on their browsing and engagement patterns, revealing heterogeneous purchase dynamics.
*   **Advanced Survival Analysis**: Utilization of Kaplan-Meier curves and Cox Proportional Hazards models to formally quantify conversion probabilities over time and across sessions, and to pinpoint significant drivers of purchase speed.
*   **Delayed Co-purchasing Behavior**: Examination of how items viewed together are later purchased together, exploring immediate vs. delayed co-purchase probabilities and the influence of category and price similarity.
*   **Key Results & Business Recommendations**: Synthesis of all findings into actionable insights and strategic recommendations for optimizing conversion funnels, retargeting strategies, and overall customer experience.

## **0. Loading data and libraries**

In [ ]:
import pandas as pd
import numpy as np
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

import re


In [ ]:
# Load the dataset
df = pd.read_csv(
    "IMA2026_Versace_Time_to_Purchase.csv",
    dtype={"user_pseudo_id": "string", "event_timestamp": "string"}
)

In [ ]:
df['user_pseudo_id'] = df['user_pseudo_id'].astype('string')

## **1. Data Cleaning & EDA**




This section of the notebook focuses on **Data Cleaning and Exploratory Data Analysis (EDA)** to prepare the raw data for further analysis. Key steps included:

*   **Unique Session Identification**: Creating a `unique_session` identifier to accurately track user sessions.
*   **Missing Values Analysis and Imputation**: Identifying and handling missing data in critical columns like `search_term` (dropped due to high missingness), `item_category` (imputed where possible, remaining missing values removed), and `engagement_time_msec` (imputed based on event type).
*   **Outlier Analysis for `engagement_time_msec`**: Investigating the distribution of engagement time, noting its skewness and the concentration of high values in specific event types, deciding to retain outliers due to their behavioral significance.
*   **Duplicate Rows Check**: Understanding the nature of duplicate rows, concluding they are inherent to GA4 data structure rather than errors.
*   **Timestamp Monotonicity Check**: Ensuring event timestamps within sessions are chronologically ordered.
*   **Exploratory Data Analysis (EDA)**: Analyzing overall dataset characteristics, user loyalty (sessions per user), device usage, login status, and average engagement time per event to gain initial insights into user behavior and sales trends.

In [ ]:
print(df.columns.tolist())
print("\n---\n")
print(df['event_name'].value_counts())
print("\n---\n")
print(df.dtypes)

### **Unique Session Analysis**

We created the `unique_session` column by concatenating `user_pseudo_id` and `session_id`. This helps accurately identify distinct user sessions, especially when `session_id` might be reused ensuring each visit is uniquely tracked for analysis.

In [ ]:
#To identify all  the unique sessions(since the variable "session_id" relies on the time stamp),
#we create a variable "unique_session" that concatenates user_pseudo_id and session_id

df['unique_session'] = df['user_pseudo_id'].astype(str)+ '_' + df['session_id'].astype(str)
df[['user_pseudo_id', 'session_id', 'unique_session']].head()

In [ ]:
num_unique_sessions = df['unique_session'].nunique()
total_rows = len(df)
num_unique_user_pseudo_ids = df['user_pseudo_id'].nunique()

print(f"Number of unique sessions: {num_unique_sessions}")
print(f"Total number of records in the DataFrame: {total_rows}")
print(f"Average number of events per session: {total_rows/ num_unique_sessions:.4f}")
print(f"Number of unique pseudo IDs: {num_unique_user_pseudo_ids}")
print(f"Average number of sessions per pseudo ID: {num_unique_sessions / num_unique_user_pseudo_ids:.4f}")

#One session generates multiple events, we have an average of 57 events per session

### **Missing Values Analysis**

This section analyzes missing values across the dataset. We'll identify columns with missing data, quantify their extent, and investigate patterns to inform our imputation or handling strategy. This is crucial for maintaining data quality and ensuring robust analysis.

In [ ]:
df = df.replace(['(not set)', '', ' '], np.nan)

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = 100 * missing_values / len(df)
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage (%)': missing_percentage})

# Display columns with missing values, sorted by percentage
display(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Percentage (%)', ascending=False))

**search term missing values**

In [ ]:
#Since the "search_term" column is empty, we drop it from the df
df = df.drop('search_term', axis=1)

**item_revenue and item_category missing values**

For `item_revenue`, a check confirmed no inconsistencies (i.e., no `transaction_id` without `item_revenue`), so no specific action was required.

For `item_category`, initial missing values were identified where `item_id` was present. An attempt was made to impute categories for items with inconsistent existing data. Subsequently, the remaining rows with `item_id` but still missing `item_category` were removed to ensure data quality, as they constituted a very small portion of the dataset.

In [ ]:
# 1. Check: transaction_id is present but item_revenue is missing
inconsistent_transactions = df[
    (df['transaction_id'].notna()) &
    (df['item_revenue'].isna())
]

print(f"Inconsistent transactions (transaction_id present, item_revenue missing): {len(inconsistent_transactions) }")


# 2. Check: item_id is present but item_category is missing
inconsistent_item_categories = df[
    (df['item_id'].notna()) &
    (df['item_category'].isna())
]

num_inconsistent_item_categories = len(inconsistent_item_categories)
total_rows = len(df)
percentage_inconsistent_item_categories = (num_inconsistent_item_categories / total_rows) * 100

print(f"Number of rows with item_id present but item_category missing: {num_inconsistent_item_categories}")
print(f"Percentage of rows with item_id present but item_category missing: {percentage_inconsistent_item_categories:.2f}%")


# 3. Identify item_ids with inconsistent category information

# item_ids that have at least one valid category
item_ids_with_category = df[
    df['item_category'].notna()
]['item_id'].dropna().unique()

# item_ids that have at least one missing category
item_ids_with_missing_category = df[
    df['item_category'].isna()
]['item_id'].dropna().unique()

# item_ids appearing in both groups (i.e., inconsistent category assignment)
inconsistent_item_ids_to_fix = list(
    set(item_ids_with_category) & set(item_ids_with_missing_category)
)


if inconsistent_item_ids_to_fix:
    print(f"\nFound {len(inconsistent_item_ids_to_fix)} item_ids with inconsistent category information.")

    # Create a mapping from item_id to a known (non-missing) item_category
    category_map = (
        df[
            df['item_id'].isin(inconsistent_item_ids_to_fix) &
            df['item_category'].notna()
        ]
        .drop_duplicates(subset=['item_id'])
        .set_index('item_id')['item_category']
    )

    # Fill missing item_category values using the mapping
    df['item_category'] = df.apply(
        lambda row: category_map[row['item_id']]
        if pd.isna(row['item_category']) and row['item_id'] in category_map
        else row['item_category'],
        axis=1
    )

    print("Missing item_category values have been filled for inconsistent item_ids.")

    # Re-check inconsistencies after filling
    re_inconsistent_item_categories = df[
        (df['item_id'].notna()) &
        (df['item_category'].isna())
    ]

    num_re_inconsistent_item_categories = len(re_inconsistent_item_categories)

    print(f"Number of rows with item_id present but item_category still missing after fix: {num_re_inconsistent_item_categories}")

else:
    print("\nNo item_ids with inconsistent category information were found.")

In [ ]:
print(f"Number of unique item_ids among the 335 rows: {re_inconsistent_item_categories['item_id'].nunique()}")
print("Top 10 item_ids with missing c.ategories:")
display(re_inconsistent_item_categories['item_id'].value_counts().head(10))

We've identified 335 rows where `item_id` is present but `item_category` remains missing, even after attempting to impute from consistent entries. Given this represents a very small fraction of the total dataset (approx. 0.11%), and to ensure data quality for downstream analysis, we will now remove these rows. This ensures that any analysis involving `item_category` will be based on complete and consistent information.

In [ ]:
initial_rows = len(df)
df = df.drop(re_inconsistent_item_categories.index)
rows_removed = initial_rows - len(df)

print(f"Removed {rows_removed} rows where 'item_id' was present but 'item_category' was missing, as previously discussed.")
print(f"DataFrame now has {len(df)} rows.")

**engagement_time_msec missing values**

Let's investigate the missingness in the `engagement_time_msec` column to understand if it's random or if there's a pattern associated with other features.

In [ ]:
# Calculate the percentage of missing values in 'engagement_time_msec'
missing_engagement_time_percentage = df['engagement_time_msec'].isnull().sum() / len(df) * 100
print(f"Percentage of missing 'engagement_time_msec': {missing_engagement_time_percentage:.2f}%")

# Separate the DataFrame into two groups: with and without missing 'engagement_time_msec'
df_missing_engagement = df[df['engagement_time_msec'].isnull()]
df_not_missing_engagement = df[df['engagement_time_msec'].notnull()]

print("\n--- Analysis for rows where 'engagement_time_msec' is MISSING ---")
print("Top event names:")
print(df_missing_engagement['event_name'].value_counts().head())
print("Top device categories:")
print(df_missing_engagement['device_category'].value_counts().head())
print("Login status distribution:")
print(df_missing_engagement['login_status'].value_counts().head())

# Display the % of missing data in the engagement time per event
print("\n--- Percentage of missing 'engagement_time_msec' per event ---")
missing_per_event = df.groupby('event_name')['engagement_time_msec'].apply(lambda x: x.isnull().sum())
total_per_event = df.groupby('event_name')['engagement_time_msec'].count() + missing_per_event
percentage_missing_per_event = (missing_per_event / total_per_event * 100).fillna(0).sort_values(ascending=False)
print(percentage_missing_per_event)

A significant portion (~9.63%) of `engagement_time_msec` values are missing.
Missingness is concentrated in 'session_start' and 'first_visit' events,
where engagement duration might not be applicable.
This pattern suggests a structural missingness rather than random.


Given the observed patterns of missingness in `engagement_time_msec`, we will adopt a two-pronged imputation strategy:

*   For `session_start` and `first_visit` events, missing values will be replaced with 0. This is because these events mark the *initiation* of an interaction, and engagement duration is often considered instantaneous or non-applicable at these points.
*   For all other event types, missing `engagement_time_msec` values will be imputed with the median engagement time specific to that event type. This approach leverages the typical duration for each event, preserving the overall distribution while being robust to outliers.

In [ ]:
# 1. Impute 'engagement_time_msec' for 'session_start' and 'first_visit' with 0
df.loc[
    (df['engagement_time_msec'].isna()) &
    (df['event_name'].isin(['session_start', 'first_visit'])),
    'engagement_time_msec'
] = 0

# 2. Calculate the median 'engagement_time_msec' for each event type (excluding 'session_start' and 'first_visit' as they are handled)
median_engagement_time_by_event = df[
    ~df['event_name'].isin(['session_start', 'first_visit'])
].groupby('event_name')['engagement_time_msec'].median()

# 3. Impute remaining missing 'engagement_time_msec' values with the median of their respective event type
df['engagement_time_msec'] = df.apply(
    lambda row:
        median_engagement_time_by_event[row['event_name']]
        if pd.isna(row['engagement_time_msec']) and row['event_name'] in median_engagement_time_by_event
        else row['engagement_time_msec'],
    axis=1
)

# Verify that there are no more missing values in 'engagement_time_msec'
missing_engagement_after_imputation = df['engagement_time_msec'].isnull().sum()
print(f"Missing 'engagement_time_msec' after imputation: {missing_engagement_after_imputation}")

**device_model missing values**

Since a huge portion (41.72%) of our data has missing values for this variable, we will not consider it in our analysis

**item_price, item_quantity, cart_value_estimate missing values**

In [ ]:
#Analysis for rows where item_price, item_quantity, and cart_value_estimate are MISSING

# Check if item_price, item_quantity, and cart_value_estimate are missing in the same places
is_item_price_missing = df['item_price'].isnull()
is_item_quantity_missing = df['item_quantity'].isnull()
is_cart_value_estimate_missing = df['cart_value_estimate'].isnull()

all_missing_same_places = (is_item_price_missing == is_item_quantity_missing).all() and \
                          (is_item_price_missing == is_cart_value_estimate_missing).all()

print(f"Are item_price, item_quantity, and cart_value_estimate missing in the same places? {all_missing_same_places}")

# Identify rows where these item-related data are missing
df_missing_item_data = df[is_item_price_missing]

# Display the count of event_name for these rows
print("\nTop event names when item-related data is missing (Absolute Count):")
print(df_missing_item_data['event_name'].value_counts())

# Compute the percentage of each event_name within the missing item data
print("\nPercentage of each event type when item-related data is missing:")
print(df_missing_item_data['event_name'].value_counts(normalize=True) * 100)

# Display some sample rows to get a better context
print("\nSample rows where item-related data is missing:")
display(df_missing_item_data.head())

In [ ]:
# Check if item_id is present where item_price, item_quantity, or cart_value_estimate are missing
# We already know from 'all_missing_same_places' that if one is missing, all three are.
# So, checking 'item_price' is sufficient.

rows_with_missing_item_details_but_present_item_id = df[
    df['item_price'].isnull() & df['item_id'].notna()
]

num_such_rows = len(rows_with_missing_item_details_but_present_item_id)

print(f"Number of rows where item price/quantity/cart value is missing but item_id is present: {num_such_rows}")

if num_such_rows > 0:
    print("Sample of these rows:")
    display(rows_with_missing_item_details_but_present_item_id.head())
else:
    print("No such rows found.")

For `item_price`, `item_quantity`, and `cart_value_estimate`, we adopted a nuanced approach to missing values.

Missing values in events like `session_start` or `login` were considered structural, as these fields are not applicable. These rows were retained, as their missingness does not indicate a data quality issue.

However, for critical e-commerce events (e.g., `view_cart`, `pay_now`, `select_item`), where product information is essential, the absence of data indicated a low data quality issue.

To maintain data integrity, 449 rows (approx. 0.15% of the dataset) corresponding to these product-related events with missing item data were removed. This ensures our analysis is based on reliable product information.

In [ ]:
# Define critical_item_events if it's not guaranteed to be globally available
critical_item_events = [
    'remove_from_cart', 'pay_now',  'add_to_wishlist',  'select_item'
]

# Identify rows with missing item data in critical events directly from df
# Assuming 'item_price' missing implies 'item_quantity' and 'cart_value_estimate' are also missing.
indices_to_remove = df[
    df['item_price'].isnull() & df['event_name'].isin(critical_item_events)
].index

initial_rows_before_removal_product = len(df)
df = df.drop(indices_to_remove, errors='ignore') # Use errors='ignore' for idempotence
rows_removed_product = initial_rows_before_removal_product - len(df)

print(f"Removed {rows_removed_product} rows due to missing product data in critical events.")
print(f"DataFrame now has {len(df)} rows.")

### **Outliers Analysis for engagement_time_msec**

In [ ]:
# Plotting the distribution and identifying outliers for 'engagement_time_msec'

plt.figure(figsize=(8, 5))
sns.histplot(df['engagement_time_msec'], bins=50, kde=True)
plt.title('Distribution of engagement_time_msec')
plt.xlabel('engagement_time_msec')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['engagement_time_msec'])
plt.title('Box Plot of engagement_time_msec to visualize outliers')
plt.xlabel('engagement_time_msec')
plt.grid(True)
plt.show()


print("\nDescriptive statistics for engagement_time_msec:")
print(df['engagement_time_msec'].describe())

Q1 = df['engagement_time_msec'].quantile(0.25)
Q3 = df['engagement_time_msec'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['engagement_time_msec'] < lower_bound) | (df['engagement_time_msec'] > upper_bound)]
print(f"\nNumber of outliers detected using IQR method for engagement_time_msec: {len(outliers)}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")


In [ ]:
# Calculate the 97.5th percentile for engagement_time_msec
pct_97 = df['engagement_time_msec'].quantile(0.975)
print(f"The 97.5 th percentile for engagement_time_msec is: {pct_97:.2f}")

# Filter the DataFrame to include only the outliers (above 97th percentile)
df_outliers = df[df['engagement_time_msec'] > pct_97]
print(f"Number of outliers (above 99th percentile): {len(df_outliers)}")
print(f"Percentage of data considered as outliers: {len(df_outliers) / len(df) * 100:.2f}%")

print("\n--- Analysis of Outliers (engagement_time_msec > 97.5th percentile) ---")

# Analyze event_name distribution in outliers
print("Top event names in outliers:")
print(df_outliers['event_name'].value_counts(normalize=True).head(10) * 100)

# Analyze device_category distribution in outliers
print("\nTop device categories in outliers:")
print(df_outliers['device_category'].value_counts(normalize=True).head() * 100)

# Analyze login_status distribution in outliers
print("\nLogin status distribution in outliers:")
print(df_outliers['login_status'].value_counts(normalize=True).head() * 100)


In [ ]:
event_outlier_counts = df_outliers['event_name'].value_counts()
total_event_counts = df['event_name'].value_counts()

# Calculate the percentage of outliers per event
percentage_outliers_per_event = (event_outlier_counts / total_event_counts) * 100

print("\nPercentage of Outliers per Event Type:")
display(percentage_outliers_per_event.sort_values(ascending=False).dropna())

The distribution of engagement_time_msec is highly right-skewed, with some extreme values.
These values seem to be associated with specific high-intent events (e.g., contact us, add_to_cart, login).
Engagement time varies significantly across event types, so global outlier removal would discard meaningful behavioral signals.
For this reason, we retain all observations and we will eventually account for this heterogeneity in the modeling stage.

### **Additional Checks**

**duplicate rows checks**

In [ ]:
#Duplicate rows check

# Count total number of rows in the dataset
total_rows = len(df)

# Count duplicated rows (exact duplicates across all columns)
duplicate_rows = df.duplicated().sum()

# Compute number of unique rows
unique_rows = total_rows - duplicate_rows

# Display results in a clean and readable format
print(f"Total number of rows:        {total_rows}")
print(f"Number of duplicate rows:   {duplicate_rows}")
print(f"Number of unique rows:      {unique_rows}")

# Display an example of duplicate rows
if duplicate_rows > 0:
    print("\nExample of duplicate rows (first 5):")
    display(df[df.duplicated(keep=False)].head())

The large number of apparent duplicate rows is likely due to the structure of the GA4 export rather than to actual data errors. In particular, events such as view_item_list can be repeated multiple times within the same timestamp, session, and page because one single list-view event may be associated with several products displayed on the page. In our sample of duplicates, most fields are identical, while product-related information such as item_price changes and item_id is often missing. This suggests that these rows may represent multiple item impressions generated from the same browsing event, rather than true accidental duplicates.

**Timestamp Monotonicity Check**

A **Timestamp Monotonicity Check** verifies events within a session are in chronological order. This ensures data integrity and accurate sequence analysis, crucial for understanding user journeys and calculating durations like 'Time to Purchase'.

In [ ]:
#Timestamp monotonicity check

# Convert timestamp to numeric format (if not already)
df['event_timestamp_num'] = df['event_timestamp'].astype(float)

# Sort data by session and timestamp
df_sorted = df.sort_values(['unique_session', 'event_timestamp_num'])

# Check for non-monotonic sequences within each session
non_monotonic_sessions = (
    df_sorted
    .groupby('unique_session')['event_timestamp_num']
    .apply(lambda x: (x.diff().dropna() < 0).any())
)

# Count sessions with at least one timestamp inconsistency
num_non_monotonic = non_monotonic_sessions.sum()
total_sessions = len(non_monotonic_sessions)

# Formatter with dot as thousands separator
def format_with_dots(x):
    return f"{x:,}".replace(",", ".")

# Display results
print(f"Sessions with non-monotonic timestamps: {format_with_dots(num_non_monotonic)}")
print(f"Total number of sessions:               {format_with_dots(total_sessions)}")

### **Category Cleaning**

The original dataset consisted of raw, nested strings (e.g., jeans couture:men:clothing:pants).To make this data actionable, we performed a data engineering process.

We developed a custom parsing logic to "disaggregate" these strings into three distinct, standardized dimensions:

- Target: The demographic audience (Men, Women, Children, Home and Lifestyle).
- Typology: The macro-merchandise category (Clothing, Shoes, Accessories, etc.).
- Product: The specific end-product detail.


In [ ]:
def parse_categories_refined(cat_string):
    if pd.isna(cat_string) or cat_string in ["(not set)", "n/a", ""]:
        return pd.Series(["n/a", "n/a", "n/a"])

    # Split the hierarchy: women:clothing:blouses and tops
    parts = [p.strip().lower() for p in str(cat_string).split(':')]

    # Define what constitutes a "Target" in the Versace context
    valid_targets = ['women', 'men', 'children', 'home and lifestyle', 'unisex']

    # 1. Identify Target: find the first part that matches our valid list
    target = "unclassified"
    for part in parts:
        if part in valid_targets:
            target = part
            break

    # 2. Identify Typology: the part that is NOT the target
    # If the first part is the target, typology is the second. Otherwise, first is typology.
    typology = "n/a"
    if len(parts) > 1:
        typology = parts[1] if parts[0] == target else parts[0]
    elif len(parts) == 1 and parts[0] not in valid_targets:
        typology = parts[0]

    # 3. Identify Product: always the most specific part (the last one)
    product = parts[-1] if len(parts) > 0 else "n/a"

    # Cleaning: if product is the same as typology, we keep it but acknowledge the hierarchy
    return pd.Series([target.capitalize(), typology.capitalize(), product.capitalize()])

# Applying the refined parsing to the main df
df[['Target', 'Typology', 'Product']] = df['item_category'].apply(parse_categories_refined)

df.head()

### **Exploratory Data Analysis**

In this phase we analyzed:
* **Sales Trends:** Daily transaction volume and growth patterns.
* **User Loyalty:** Session frequency and retention segments.
* **User Profiles:** Device usage and login authentication rates.
* **Engagement:** Average time spent on key customer journey events.



In [ ]:
# Ensure event_datetime is in datetime format and unique_session is created
if 'event_datetime' not in df.columns:
    # Convert event_timestamp to numeric first to avoid FutureWarning
    df['event_datetime'] = pd.to_datetime(df['event_timestamp'].astype(int), unit='us')
if 'unique_session' not in df.columns:
    df['unique_session'] = df['user_pseudo_id'].astype(str) + '_' + df['session_id'].astype(str)

# 1. Time frame of the data set
min_date = df['event_datetime'].min().strftime('%Y-%m-%d %H:%M:%S')
max_date = df['event_datetime'].max().strftime('%Y-%m-%d %H:%M:%S')
print(f"Time frame of the data set: From {min_date} to {max_date}")
print(f"Data spans approximately {(df['event_datetime'].max() - df['event_datetime'].min()).days} days.\n")

# 2. Percentage of unique sessions where user logged in
total_unique_sessions = df['unique_session'].nunique()
unique_logged_in_sessions = df[df['login_status'] == 'loggedin']['unique_session'].nunique()
percentage_logged_in_sessions = (unique_logged_in_sessions / total_unique_sessions) * 100
print(f"Percentage of unique sessions where user logged in: {percentage_logged_in_sessions:.2f}%\n")

# 3. For each user_id the number of user_pseudo_id: compute the mean of this value
# Consider only rows where a user_id is present (i.e., logged in) to analyze registered user behavior
df_with_user_id = df[df['user_id'].notna()]
if not df_with_user_id.empty:
    user_pseudo_id_counts_per_user_id = df_with_user_id.groupby('user_id')['user_pseudo_id'].nunique()
    mean_user_pseudo_id_per_user = user_pseudo_id_counts_per_user_id.mean()
    print(f"Mean number of user_pseudo_id per user_id (for registered users): {mean_user_pseudo_id_per_user:.2f}\n")
else:
    print("No registered users found to calculate mean user_pseudo_id per user_id.\n")

# 4. Number of transactions in the DB
num_transactions = df['transaction_id'].nunique() # Counts unique non-null transaction_ids
print(f"Number of unique transactions in the database: {num_transactions}\n")

# 5. Percentage of events with transactions / all events
total_events = len(df)
events_with_transaction = df['transaction_id'].notna().sum()
percentage_events_with_transactions = (events_with_transaction / total_events) * 100
print(f"Percentage of events with transactions: {percentage_events_with_transactions:.2f}%\n")

# 6. Percentage of unique sessions with transactions / all unique sessions
total_unique_sessions = df['unique_session'].nunique()
sessions_with_transaction = df.loc[df['transaction_id'].notna(), 'unique_session'].nunique()
percentage_sessions_with_transactions = (sessions_with_transaction / total_unique_sessions) * 100
print(f"Percentage of unique sessions with transactions: {percentage_sessions_with_transactions:.2f}%\n")

# 7. Percentage of unique user pseudo IDs that make a transaction
total_unique_user_pseudo_ids = df['user_pseudo_id'].nunique()
user_pseudo_ids_with_transaction = df[df['transaction_id'] != "NaN"]['user_pseudo_id'].nunique()
percentage_user_pseudo_ids_with_transactions = (user_pseudo_ids_with_transaction / total_unique_user_pseudo_ids) * 100
print(f"Percentage of unique user pseudo IDs with transactions: {percentage_user_pseudo_ids_with_transactions:.2f}%\n")

# Number of users
num_unique_user_pseudo_ids = df['user_pseudo_id'].nunique()
print(f"Number of unique user pseudo IDs (unique browsing entities): {num_unique_user_pseudo_ids}")

num_unique_registered_users = df['user_id'].nunique()
print(f"Number of unique registered user IDs (logged-in users): {num_unique_registered_users}\n")

# Number of unique page locations (numerical as requested)
num_unique_page_locations = df['page_location'].nunique()
print(f"Number of unique page locations: {num_unique_page_locations}\n")

# Number of unique countries
num_unique_countries = df['country_long'].nunique()
print(f"Number of unique countries: {num_unique_countries}\n")
print(f"List of unique countries:{df['country_long'].unique() }")

This initial phase establishes the **scope and scale** of the dataset. We are analyzing **60 days** of traffic (June–July 2025) localized entirely to **Italy**. The goal is to audit user identity, session quality, and the baseline conversion rate.

* **High Session Intent:** While total event conversion is low (0.46%), a remarkable **18.75% of sessions** end in a transaction.
* **Identification Gap:** Only **12.82% of sessions** are logged in. The majority of the audience (87%) remains anonymous, limiting personalized marketing.
* **Platform Scale:** With **990 transactions** and nearly **7,000 unique URLs** visited, the site manages a high-volume catalog for a focused Italian audience.


In [ ]:
sns.set_style("whitegrid")

# Number of transactions over time (date)
df_transactions = df[df['transaction_id'].notna()].copy()
if not df_transactions.empty:
    df_transactions['event_date_only'] = df_transactions['event_datetime'].dt.date
    transactions_per_day = df_transactions.groupby('event_date_only')['transaction_id'].nunique()

    plt.figure(figsize=(12, 6))
    transactions_per_day.plot(kind='line', marker='o', linestyle='-', color='teal')
    plt.title('Number of Unique Transactions Over Time')
    plt.xlabel('Date')
    plt.ylabel('Number of Unique Transactions')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No transactions found to plot over time.\n")




We examine daily transaction trends over the 60-day period to identify sales momentum, seasonality, and the impact of promotional activities.


* **Significant Growth:** July shows a higher transaction baseline compared to June, indicating a successful mid-summer performance boost.

* **High Volatility:** Frequent fluctuations suggest the brand is highly sensitive to specific marketing pushes or weekly shopping cycles.

In [ ]:
print("=== SESSIONS PER USER  ===")

# Compute number of unique sessions per user
sessions_per_user = df.groupby('user_pseudo_id')['unique_session'].nunique()

# Formatter with dot as thousands separator
def format_with_dots(x):
    return f"{x:,}".replace(",", ".")

# Summary statistics
summary_stats = sessions_per_user.describe().round(2)

print("\nSummary statistics (sessions per user):")
print(summary_stats)

# Key segments
users_1_session = (sessions_per_user == 1).sum()
users_2plus_sessions = (sessions_per_user > 1).sum()
users_5plus_sessions = (sessions_per_user >= 5).sum()

total_users = len(sessions_per_user)

# Display segmented insights
print("\nUser segmentation based on session activity:")
print(f"Users with 1 session:        {format_with_dots(users_1_session)} ({users_1_session / total_users:.1%})")
print(f"Users with 2+ sessions:      {format_with_dots(users_2plus_sessions)} ({users_2plus_sessions / total_users:.1%})")
print(f"Users with 5+ sessions:      {format_with_dots(users_5plus_sessions)} ({users_5plus_sessions / total_users:.1%})")

This section segments users based on their **session frequency**. We aim to differentiate between "one-time" visitors and loyal "power users" to understand the brand’s retention capabilities.


* **Strong Retention:** **73.4%** of users return for a second session, and **35.9%** (the "Power Users") visit 5 or more times.
* **Engagement Outliers:** While the median is 3 sessions, the maximum of **149 sessions** indicates a small group of extremely highly-engaged users (possibly professional buyers).
* **Low Bounce Profile:** Only **26.6%** of users visit just once, which is low for e-commerce, suggesting high brand stickiness.

In [ ]:
#Studying Sessions and Users
fig, axes = plt.subplots(1, 3, figsize=(20, 9))

# Plot 1: Distribution of sessions per user
sessions_per_user_clipped = sessions_per_user.clip(upper=10)
sessions_per_user_clipped.value_counts().sort_index().plot(
    kind="bar",
    ax=axes[0],
    color="steelblue",
    edgecolor="white"
)
axes[0].set_title("Sessions per User Distribution (10+ capped)")
axes[0].set_xlabel("Number of Sessions")
axes[0].set_ylabel("Number of Users")
axes[0].tick_params(axis="x", rotation=0)

# Plot 2: Unique sessions by device category
device_counts = df.groupby("device_category")["unique_session"].nunique()
axes[1].pie(
    device_counts.values,
    labels=device_counts.index,
    autopct="%1.1f%%",
    colors=["steelblue", "coral", "mediumseagreen"]
)
axes[1].set_title("Unique Sessions by Device Category")

# Plot 3: Unique sessions by login status
login_counts = df.groupby("login_status")["unique_session"].nunique()
axes[2].pie(
    login_counts.values,
    labels=login_counts.index,
    autopct="%1.1f%%",
    colors=["steelblue", "coral"]
)
axes[2].set_title("Unique Sessions by Login Status")

plt.tight_layout()
plt.show()

We visualize the **user profile and behavior** through session distribution, device preference, and authentication status. This provides a 360-degree view of how and where users interact with the brand.
* **Engagement "Bump":** The session distribution bar chart shows a high volume of users returning **10+ times** (capped), highlighting a distinct segment of dedicated repeat visitors.
* The main **device** used by users is mobile.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Prepare the data
min_session_numbers = df.groupby('user_pseudo_id')['ga_session_number'].min().reset_index()

# Define a cap for visualization to avoid the "long tail" squeezing the chart
cap_limit = 30
plot_data = min_session_numbers['ga_session_number'].copy()
plot_data_capped = np.where(plot_data > cap_limit, cap_limit + 1, plot_data)

# 2. Create the Plot
plt.figure(figsize=(14, 6))
sns.set_style("whitegrid")

# Main Histogram
ax = sns.histplot(plot_data_capped,
                  bins=range(1, cap_limit + 3),
                  kde=False,
                  color='#003366', # Versace-style deep blue
                  edgecolor='white',
                  alpha=0.8)

# 3. Customizing the X-axis labels to show the "Cap"
xticks = list(range(1, cap_limit + 1, 2)) + [cap_limit + 1]
xtick_labels = [str(i) for i in range(1, cap_limit + 1, 2)] + [f'>{cap_limit}']
plt.xticks(xticks, xtick_labels)

# 4. Add Secondary Axis for Cumulative Percentage (Pareto-style)
ax2 = ax.twinx()
counts, bin_edges = np.histogram(plot_data_capped, bins=range(1, cap_limit + 3))
cumulative = np.cumsum(counts) / len(plot_data) * 100
ax2.plot(range(1, cap_limit + 2), cumulative, color='#C5A059', marker='o', label='Cumulative %') # Gold color

# 5. Annotations & Aesthetics
ax.set_title("User Entry Point: Distribution of First Observed Session", fontsize=16, pad=20)
ax.set_xlabel("Starting Session Number", fontsize=12)
ax.set_ylabel("Number of Users", fontsize=12)
ax2.set_ylabel("Cumulative Percentage of Users (%)", fontsize=12, color='#C5A059')
ax2.set_ylim(0, 110)

# Add a label for the "Session 1" bar specifically
session_1_count = (min_session_numbers['ga_session_number'] == 1).sum()
session_1_pct = (session_1_count / len(min_session_numbers)) * 100
ax.text(1.5, session_1_count, f' {session_1_pct:.1f}% New Users',
        va='bottom', fontweight='bold', color='#003366')

plt.tight_layout()
plt.show()

- The graph shows a highly skewed distribution, where the vast majority of users begin their activity in the very first session.
- This is reinforced by the **high proportion of new users** (around 75%+), indicating that most engagement happens immediately upon entry.
- After the first session, the number of users drops sharply, suggesting that relatively few users start interacting at later stages.
- The cumulative percentage curve increases very quickly, confirming that most users are captured early in their lifecycle.

In [ ]:
#Average engagement time for event
# Calculate event counts
event_counts = df['event_name'].value_counts()

# Filter for events with at least 100 occurrences: to avoid outliers due to the contact us and sign_up events
events_to_plot = event_counts[event_counts >= 100].index
df_filtered = df[df['event_name'].isin(events_to_plot)]

# Average engagement time per event for filtered events
avg_engagement_time_per_event = df_filtered.groupby('event_name')['engagement_time_msec'].mean().sort_values(ascending=False)
if not avg_engagement_time_per_event.empty:
    plt.figure(figsize=(12, 7))
    sns.barplot(x=avg_engagement_time_per_event.index, y=avg_engagement_time_per_event.values, hue=avg_engagement_time_per_event.index, legend=False)
    plt.title('Average Engagement Time per Event Type (Events with >= 100 Occurrences)')
    plt.xlabel('Event Name')
    plt.ylabel('Average Engagement Time (msec)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No engagement time data found for events with at least 100 occurrences to plot.\n")

We analyze **engagement time per event** to understand which parts of the customer journey require the most consideration. This helps distinguish between quick transactional actions and "high-interest" browsing activities.


* **Wishlist Intensity:** `add_to_wishlist` has the highest engagement time (approx. 26 seconds). This suggests users spend time contemplating items before saving them.
* **Selection Process:** `select_item` and `login` also show high engagement times, indicating a deliberate selection process and a potentially multi-step authentication phase.
* **Low Engagement at Conversion:** Events like `purchase`, `view_cart`, and `begin_checkout` show near-zero engagement time in this view. This suggests that once a user decides to buy, the actual technical process is extremely fast.

Overall in this phase:

* **Sales Momentum**: We identified a significant performance boost in July, with a major peak early in the month where transactions nearly tripled compared to June.

* **High Brand Stickiness**: Retention is strong; 73.4% of users return for at least a second session, and over a third (35.9%) are high-frequency "Power Users."

* **Mobile Dominance**: The audience is overwhelmingly Mobile-first (67%), yet largely anonymous, as 88.5% of users navigate without logging in.

* **High-Intent Micro-Moments**: The Wishlist and Item Selection phases command the highest engagement time (up to 26 seconds), while the actual checkout and purchase events are processed almost instantaneously.



The analysis indicates a highly loyal, mobile-centric audience that uses the site heavily for product consideratio. To drive growth, the business should focus on optimizing the mobile UX, and implementing incentives for user registration to capture valuable first-party data from the anonymous majority.

## **2. Time to purchase Dataframe**

The `ttp_df` (Time-to-Purchase DataFrame) was constructed at the `user_pseudo_id`–`item_id` level to model the time between a user’s first observed view of an item and its eventual purchase, while also retaining non-purchased items for survival analysis.

Here is how it was built:

1. **Restricting the data to item-level events**  
   The process begins by filtering the main dataset to observations with a valid `item_id`. This ensures that the analysis focuses on product-level interactions rather than generic website activity.

2. **Identifying the first observed item view**  
   For each `user_pseudo_id` and `item_id` pair, the earliest `view_item` event is selected. This defines the starting point of the consideration process (`view_time`).

3. **Identifying the first valid purchase after the first view**  
   Purchase events are then matched to each user-item pair, but only purchases occurring at or after the first view are considered valid. Among these, the earliest valid purchase is kept as the conversion event (`purchase_time`). This avoids illogical cases in which a purchase would appear to happen before the observed consideration period.

4. **Keeping both purchased and non-purchased user-item pairs**  
   The first-view dataset is merged with the purchase dataset using a left join, so that all viewed user-item pairs are retained. This means that:
   - pairs with a valid purchase are treated as observed conversions;
   - pairs without a purchase are kept as non-converted observations.

5. **Creating the event indicator and censoring structure**  
   An `event_observed` variable is created:
   - `1` if the item was purchased,
   - `0` if it was not purchased within the observation window.  
   
   For non-purchased pairs, the end of the dataset is used as the censoring point. This allows the dataset to be used in a proper survival-analysis framework.

6. **Computing time durations**  
   For purchased pairs, time to purchase is calculated as the difference between `purchase_time` and `view_time`, expressed in both days and minutes.  
   For survival analysis more generally, a common duration is defined from `view_time` to:
   - `purchase_time` if the event is observed,
   - the end of the observation window if the event is censored.

7. **Aggregating engagement within the relevant time window**  
   The dataset is enriched with total `engagement_time_msec`, computed by summing engagement across all events for the same user-item pair occurring between the first view and the end of the observation window (purchase time for converted pairs, censoring time for non-converted pairs).

8. **Adding GA_session_number at first visit and returning_visitor**
We enrich the dataset by adding two variables related to user activity over time. First, we include GA_session_number at first visit, which captures the session index when the user is first observed. Second, we create a returning_visitor indicator to distinguish new vs. returning users.
The variable is set to 1 if the GA_session_number > 0 (i.e., the user has already had at least one prior session), and 0 otherwise.



In [ ]:
# ============================================================
# FINAL TTP DATASET FOR SURVIVAL ANALYSIS
# - One row per user_pseudo_id-item_id pair
# - Includes purchased and non-purchased items
# - Proper right-censoring for non-purchased items
# - Adds journey depth (session number) and returning indicators
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 0. PREP
# ------------------------------------------------------------
df = df.copy()
df["event_datetime"] = pd.to_datetime(df["event_datetime"])

# Extract the very first session number observed for each user in the entire dataset
user_initial_session = (
    df.groupby("user_pseudo_id")["ga_session_number"]
    .min()
    .reset_index()
    .rename(columns={"ga_session_number": "ga_session_number_at_first_visit"})
)

df_items = (
    df.loc[df["item_id"].notna()].copy()
    .sort_values(["user_pseudo_id", "item_id", "event_datetime"])
)

end_of_observation = df["event_datetime"].max()

# ------------------------------------------------------------
# 1. FIRST VIEW PER USER-ITEM
# ------------------------------------------------------------
# Removed 'campaign' from the columns list
first_view_cols = [
    "user_pseudo_id", "item_id", "event_datetime",
    "device_category", "item_category", "item_price"
]

# Add optional columns only if they exist
optional_first_view_cols = []
for col in ["login_status", "session_id", "has_utm", "utm_source", "utm_medium", "utm_campaign", "campaign_name"]:
    if col in df_items.columns:
        optional_first_view_cols.append(col)

first_views = (
    df_items.loc[df_items["event_name"] == "view_item", first_view_cols + optional_first_view_cols]
    .drop_duplicates(["user_pseudo_id", "item_id"], keep="first")
    .rename(columns={
        "event_datetime": "view_time",
        "device_category": "view_device",
        "item_price": "view_item_price",
        "login_status": "first_login_status",
        "session_id": "first_view_session_id",
        "has_utm": "first_view_has_utm",
        "utm_source": "first_view_utm_source",
        "utm_medium": "first_view_utm_medium",
        "utm_campaign": "first_view_utm_campaign",
        "campaign_name": "first_view_campaign_name"
    })
    .copy()
)

# ------------------------------------------------------------
# 2. FIRST VALID PURCHASE AFTER FIRST VIEW
# ------------------------------------------------------------
purchase_cols = [
    "user_pseudo_id", "item_id", "event_datetime",
    "transaction_id", "item_revenue", "item_quantity", "cart_value_estimate"
]

if "session_id" in df_items.columns:
    purchase_cols.append("session_id")

purchases = (
    df_items.loc[df_items["event_name"] == "purchase", purchase_cols]
    .rename(columns={
        "event_datetime": "purchase_time",
        "session_id": "purchase_session_id"
    })
    .copy()
)

purchase_candidates = first_views[["user_pseudo_id", "item_id", "view_time"]].merge(
    purchases,
    on=["user_pseudo_id", "item_id"],
    how="left"
)

purchase_candidates = purchase_candidates.loc[
    purchase_candidates["purchase_time"].isna() |
    (purchase_candidates["purchase_time"] >= purchase_candidates["view_time"])
].copy()

first_purchases_after_view = (
    purchase_candidates
    .sort_values(["user_pseudo_id", "item_id", "purchase_time"])
    .drop_duplicates(["user_pseudo_id", "item_id"], keep="first")
    .drop(columns=["view_time"])
)

# ------------------------------------------------------------
# 3. BUILD CORE TTP DATASET
# ------------------------------------------------------------
ttp_df = first_views.merge(
    first_purchases_after_view,
    on=["user_pseudo_id", "item_id"],
    how="left"
)

# Merge the initial session info
ttp_df = ttp_df.merge(user_initial_session, on="user_pseudo_id", how="left")

# Create returning_visitor indicator
# 1 if user has had at least one prior session (min session > 1), 0 otherwise
ttp_df["returning_visitor"] = (ttp_df["ga_session_number_at_first_visit"] > 1).astype(int)

ttp_df["event_observed"] = ttp_df["purchase_time"].notna().astype(int)
ttp_df["end_time"] = ttp_df["purchase_time"].fillna(end_of_observation)

# One survival duration only: use days
ttp_df["duration_days"] = (
    (ttp_df["end_time"] - ttp_df["view_time"]).dt.total_seconds() / (3600 * 24)
)

# Add duration in minutes
ttp_df["duration_minutes"] = (
    (ttp_df["end_time"] - ttp_df["view_time"]).dt.total_seconds() / 60
)

# Descriptive TTP for purchased rows only
ttp_df["days_to_purchase"] = np.where(
    ttp_df["event_observed"] == 1,
    ttp_df["duration_days"],
    np.nan
)

ttp_df["minutes_to_purchase"] = np.where(
    ttp_df["event_observed"] == 1,
    ttp_df["duration_minutes"],
    np.nan
)

# ------------------------------------------------------------
# 4. HELPER TABLE: EVENTS WITHIN EACH ITEM'S CONSIDERATION WINDOW
# ------------------------------------------------------------
window_keys = ttp_df[["user_pseudo_id", "item_id", "view_time", "end_time"]].copy()

events_in_window = df_items.merge(
    window_keys,
    on=["user_pseudo_id", "item_id"],
    how="inner"
)

events_in_window = events_in_window.loc[
    (events_in_window["event_datetime"] >= events_in_window["view_time"]) &
    (events_in_window["event_datetime"] <= events_in_window["end_time"])
].copy()

# ------------------------------------------------------------
# 5. FEATURE ENGINEERING
# ------------------------------------------------------------

# Total engagement within the consideration window
engagement_totals = (
    events_in_window
    .groupby(["user_pseudo_id", "item_id"], as_index=False)["engagement_time_msec"]
    .sum()
)

ttp_df = ttp_df.merge(
    engagement_totals,
    on=["user_pseudo_id", "item_id"],
    how="left"
)
ttp_df["engagement_time_msec"] = ttp_df["engagement_time_msec"].fillna(0)

# Number of views of the same item within the window
n_views_item = (
    events_in_window.loc[events_in_window["event_name"] == "view_item"]
    .groupby(["user_pseudo_id", "item_id"], as_index=False)
    .size()
    .rename(columns={"size": "n_views_item"})
)

ttp_df = ttp_df.merge(n_views_item, on=["user_pseudo_id", "item_id"], how="left")
ttp_df["n_views_item"] = ttp_df["n_views_item"].fillna(0).astype(int)

# Add-to-cart variables within the window
add_to_cart_counts = (
    events_in_window.loc[events_in_window["event_name"] == "add_to_cart"]
    .groupby(["user_pseudo_id", "item_id"], as_index=False)
    .size()
    .rename(columns={"size": "n_add_to_cart"})
)

ttp_df = ttp_df.merge(add_to_cart_counts, on=["user_pseudo_id", "item_id"], how="left")
ttp_df["n_add_to_cart"] = ttp_df["n_add_to_cart"].fillna(0).astype(int)
ttp_df["add_to_cart_flag"] = (ttp_df["n_add_to_cart"] > 0).astype(int)

# Number of distinct sessions within the window
if "session_id" in events_in_window.columns:
    n_sessions = (
        events_in_window
        .groupby(["user_pseudo_id", "item_id"], as_index=False)["session_id"]
        .nunique()
        .rename(columns={"session_id": "n_sessions_before_end"})
    )

    ttp_df = ttp_df.merge(n_sessions, on=["user_pseudo_id", "item_id"], how="left")
    ttp_df["n_sessions_before_end"] = ttp_df["n_sessions_before_end"].fillna(0).astype(int)

# Number of distinct OTHER items viewed by the same user
all_user_views = df_items.loc[
    df_items["event_name"] == "view_item",
    ["user_pseudo_id", "item_id", "event_datetime"]
].copy()

user_views_with_target_context = window_keys.merge(
    all_user_views,
    on="user_pseudo_id",
    how="left",
    suffixes=("_target", "_viewed")
)

user_views_in_window = user_views_with_target_context.loc[
    (user_views_with_target_context["event_datetime"] >= user_views_with_target_context["view_time"]) &
    (user_views_with_target_context["event_datetime"] <= user_views_with_target_context["end_time"]) &
    (user_views_with_target_context["item_id_viewed"] != user_views_with_target_context["item_id_target"])
].copy()

n_distinct_other_items = (
    user_views_in_window
    .groupby(["user_pseudo_id", "item_id_target"], as_index=False)["item_id_viewed"]
    .nunique()
    .rename(columns={
        "item_id_target": "item_id",
        "item_id_viewed": "n_distinct_other_items_viewed"
    })
)

ttp_df = ttp_df.merge(
    n_distinct_other_items,
    on=["user_pseudo_id", "item_id"],
    how="left"
)
ttp_df["n_distinct_other_items_viewed"] = (
    ttp_df["n_distinct_other_items_viewed"].fillna(0).astype(int)
)

# ------------------------------------------------------------
# 5B. LOGIN AND SAME-SESSION FEATURES
# ------------------------------------------------------------

# Binary logged-in flag at first view
if "first_login_status" in ttp_df.columns:
    login_clean = (
        ttp_df["first_login_status"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    ttp_df["logged_in_first_view"] = np.where(
        login_clean.isin(["loggedin", "logged_in", "logged-in", "login", "logged in", "1", "true", "yes"]),
        1,
        0
    )
else:
    ttp_df["logged_in_first_view"] = np.nan

# Add-to-cart in the same session as the first view
if "session_id" in df_items.columns and "first_view_session_id" in ttp_df.columns:
    same_session_atc = (
        df_items.loc[df_items["event_name"] == "add_to_cart", [
            "user_pseudo_id", "item_id", "session_id", "event_datetime"
        ]]
        .merge(
            ttp_df[["user_pseudo_id", "item_id", "view_time", "first_view_session_id"]],
            on=["user_pseudo_id", "item_id"],
            how="inner"
        )
    )

    same_session_atc = same_session_atc.loc[
        (same_session_atc["session_id"] == same_session_atc["first_view_session_id"]) &
        (same_session_atc["event_datetime"] >= same_session_atc["view_time"])
    ].copy()

    add_to_cart_same_session = (
        same_session_atc
        .groupby(["user_pseudo_id", "item_id"], as_index=False)
        .size()
        .rename(columns={"size": "n_add_to_cart_same_session"})
    )

    add_to_cart_same_session["add_to_cart_same_session"] = 1

    ttp_df = ttp_df.merge(
        add_to_cart_same_session[[
            "user_pseudo_id", "item_id",
            "n_add_to_cart_same_session", "add_to_cart_same_session"
        ]],
        on=["user_pseudo_id", "item_id"],
        how="left"
    )

    ttp_df["n_add_to_cart_same_session"] = (
        ttp_df["n_add_to_cart_same_session"].fillna(0).astype(int)
    )
    ttp_df["add_to_cart_same_session"] = (
        ttp_df["add_to_cart_same_session"].fillna(0).astype(int)
    )
else:
    ttp_df["add_to_cart_same_session"] = np.nan

# Final modeling price: price at first view
ttp_df["analysis_price"] = ttp_df["view_item_price"]

# price tiers
valid_price = ttp_df["analysis_price"].notna().sum()
if valid_price >= 3:
    ttp_df["price_tier"] = pd.qcut(
        ttp_df["analysis_price"],
        q=3,
        labels=["low", "medium", "high"],
        duplicates="drop"
    )
else:
    ttp_df["price_tier"] = np.nan

# Add 'paid_entry' if 'first_view_has_utm' is present
if 'first_view_has_utm' in ttp_df.columns:
    ttp_df["paid_entry"] = ttp_df["first_view_has_utm"]
else:
    ttp_df["paid_entry"] = 0 # Default to 0 if UTM info not available

# ------------------------------------------------------------
# 6. VALIDATION CHECKS
# ------------------------------------------------------------
assert ttp_df.groupby(["user_pseudo_id", "item_id"]).size().max() == 1, "Duplicate user-item rows found."
assert (ttp_df["duration_days"] >= 0).all(), "Negative durations found."
assert (
    ttp_df.loc[ttp_df["event_observed"] == 1, "purchase_time"] >=
    ttp_df.loc[ttp_df["event_observed"] == 1, "view_time"]
).all(), "Some purchases occur before view_time."

# ------------------------------------------------------------
# 7. FINAL COLUMN ORDER
# ------------------------------------------------------------
preferred_order = [
    "user_pseudo_id", "item_id",
    "view_time", "purchase_time", "end_time", "event_observed",
    "duration_days", "duration_minutes", "days_to_purchase", "minutes_to_purchase",
    "ga_session_number_at_first_visit", "returning_visitor",
    "first_view_session_id", "purchase_session_id",
    "first_login_status", "logged_in_first_view",
    "view_device", "paid_entry",
    "item_category", "view_item_price", "analysis_price", "price_tier",
    "item_revenue", "item_quantity", "cart_value_estimate", "transaction_id",
    "engagement_time_msec",
    "n_views_item",
    "n_add_to_cart", "add_to_cart_flag",
    "add_to_cart_same_session",
    "n_sessions_before_end",
    "n_distinct_other_items_viewed"
]

existing_cols = [c for c in preferred_order if c in ttp_df.columns]
remaining_cols = [c for c in ttp_df.columns if c not in existing_cols]
ttp_df = ttp_df[existing_cols + remaining_cols]

ttp_df.head()

In [ ]:
print("--- Data Integrity Checks for ttp_df ---")

# 1. Check for one row per user_pseudo_id-item_id
unique_user_item_pairs = ttp_df.groupby(['user_pseudo_id', 'item_id']).size()
num_duplicate_user_item_pairs = (unique_user_item_pairs > 1).sum()

if num_duplicate_user_item_pairs == 0:
    print("Verification 1 (Unique User-Item Pairs): PASSED - Each user_pseudo_id-item_id pair has exactly one row.")
else:
    print(f"Verification 1 (Unique User-Item Pairs): FAILED - Found {num_duplicate_user_item_pairs} user_pseudo_id-item_id pairs with more than one row.")
    # Optionally, display the problematic pairs:
    # print(unique_user_item_pairs[unique_user_item_pairs > 1])

# 2. Check if first_view always precedes purchase
# The 'view_time' column in ttp_df represents the first_view_time
# The 'purchase_time' column represents the purchase time

inconsistent_purchases = ttp_df[
    (ttp_df['purchase_time'] < ttp_df['view_time'])
]

if len(inconsistent_purchases) == 0:
    print("Verification 2 (First View Precedes Purchase): PASSED - For all purchased items, purchase_time >= view_time.")
else:
    print(f"Verification 2 (First View Precedes Purchase): FAILED - Found {len(inconsistent_purchases)} cases where purchase_time < view_time.\n")
    print("Sample of inconsistent purchases:")
    display(inconsistent_purchases[['user_pseudo_id', 'item_id', 'view_time', 'purchase_time']].head())

print("--- End of Data Integrity Checks ---")

In [ ]:
#we then add category division

# Applying the refined parsing to ttp_df
ttp_df[['Target', 'Typology', 'Product']] = ttp_df['item_category'].apply(parse_categories_refined)

### **Time to Purchase Visual Inspection**

In [ ]:
#Summary & Distribution
def get_ttp_summary(df_in, user_col='user_pseudo_id'):
    # Create Buckets
    bins = [-1, 1/24, 1, 7, float('inf')]
    labels = ['< 1 Hour', '1-24 Hours', '1-7 Days', '7+ Days']
    df_in['time_bucket'] = pd.cut(df_in['days_to_purchase'], bins=bins, labels=labels)

    # Filter out NaN user IDs if user_col is 'user_id'
    if user_col == 'user_id':
        unique_users_df = df_in[df_in[user_col].notna()]
    else:
        unique_users_df = df_in

    summary = {
        "Total Purchases": df_in['event_observed'].sum(),
        f"Unique Users ({user_col})": unique_users_df[user_col].nunique(),
        "Mean Days": df_in["days_to_purchase"].mean(),
        "Median Days": df_in["days_to_purchase"].median(),
        "Mean Minutes": df_in["days_to_purchase"].mean()*1440, # Mean in minutes added
        "Median Minutes": df_in["days_to_purchase"].median()*1440, # Median in minutes added
        "Avg Engagement (Min)": (df_in["engagement_time_msec"].mean() / 60000) # Average engagement time in minutes
    }
    dist = df_in['time_bucket'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%'
    return pd.Series(summary), dist

# Original analysis with user_pseudo_id
summary_pseudo, distribution_pseudo = get_ttp_summary(ttp_df, user_col='user_pseudo_id')
print("\n--- TTP Summary Metrics (User Pseudo ID) ---")
print(summary_pseudo)
print("\n--- Purchase Distribution (User Pseudo ID) ---")
print(distribution_pseudo)



In [ ]:
# Set a style for the plots
sns.set_style("whitegrid")

print("\n--- Visualizing Time to Purchase (TTP) Distribution ---")

# 1. Histogram of minutes_to_purchase
plt.figure(figsize=(12, 6))
sns.histplot(ttp_df['minutes_to_purchase'], bins=50, kde=True)
plt.title('Distribution of Time to Purchase (Minutes)')
plt.xlabel('Minutes to Purchase')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 2. Box Plot of minutes_to_purchase to visualize outliers
plt.figure(figsize=(12, 6))
sns.boxplot(x=ttp_df['minutes_to_purchase'])
plt.title('Box Plot of Time to Purchase (Minutes)')
plt.xlabel('Minutes to Purchase')
plt.grid(True)
plt.show()

# 3. Bar chart of time_bucket distribution
plt.figure(figsize=(10, 6))
sns.barplot(x=ttp_df['time_bucket'].value_counts().index, y=ttp_df['time_bucket'].value_counts(normalize=True).values * 100, palette='viridis', hue=ttp_df['time_bucket'].value_counts().index, legend=False)
plt.title('Distribution of Time to Purchase Buckets')
plt.xlabel('Time Bucket')
plt.ylabel('Percentage of Purchases')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\n--- TTP Descriptive Statistics ---")
print(ttp_df['minutes_to_purchase'].describe())

print("\n--- TTP vs. Total Engagement Time ---")
# Scatter plot of Log(TTP) vs. Log(Engagement Time)
plt.figure(figsize=(12, 7))
sns.scatterplot(
    x=np.log1p(ttp_df['engagement_time_msec'] / 1000), # Convert to seconds and then log
    y=np.log1p(ttp_df['minutes_to_purchase']),
    hue='view_device',
    data=ttp_df,
    alpha=0.6,
    s=50
)
plt.title('Log(TTP in Minutes) vs. Log(Engagement Time in Seconds) by View Device')
plt.xlabel('Log(Total Engagement Time in Seconds + 1)')
plt.ylabel('Log(Time to Purchase in Minutes + 1)')
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

# Calculate and print correlation
correlation = ttp_df[['minutes_to_purchase', 'engagement_time_msec']].corr().loc['minutes_to_purchase', 'engagement_time_msec']
print(f"Correlation between Time to Purchase (Minutes) and Total Engagement Time (msec): {correlation:.2f}")

In [ ]:
#here we add the login vairable at view event

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

# Ensure 'event_datetime' is correctly set in 'df'
# This check is added for robustness, as 'event_datetime' is typically created earlier.
if 'event_datetime' not in df.columns:
    df['event_datetime'] = pd.to_datetime(df['event_timestamp'].astype(int), unit='us')

# Create a temporary DataFrame with the login_status at the exact first view time for each user-item pair
# This involves filtering the original 'df' to match the 'view_time' and 'item_id' in 'ttp_df'
login_status_at_view = df[
    (df['event_name'] == 'view_item') # Only consider view events
].merge(
    ttp_df[['user_pseudo_id', 'item_id', 'view_time']],
    left_on=['user_pseudo_id', 'item_id', 'event_datetime'],
    right_on=['user_pseudo_id', 'item_id', 'view_time'],
    how='inner'
)[['user_pseudo_id', 'item_id', 'login_status']].drop_duplicates()

# Merge this login_status into ttp_df
ttp_df_with_login = ttp_df.merge(login_status_at_view, on=['user_pseudo_id', 'item_id'], how='left')

ttp_df_with_login

In [ ]:
# Handle cases where login_status might still be missing (e.g., if 'login_status' was NaN in the original df)
ttp_df_with_login['login_status'] = ttp_df_with_login['login_status'].fillna('not logged')

# Get device category at the time of the purchase event from the original df
# We consider only actual purchase events to link device category to the purchase session
purchase_device_info = df[df['event_name'] == 'purchase'][['session_id', 'device_category']].drop_duplicates()
purchase_device_info.rename(columns={'session_id': 'purchase_session_id', 'device_category': 'purchase_device'}, inplace=True)

# Merge purchase device into ttp_df_with_login. This will add purchase_device for purchased items.
ttp_df_with_login = ttp_df_with_login.merge(purchase_device_info, on='purchase_session_id', how='left')

print("\n--- TTP by Device (View) ---")
# Filter for mobile and desktop devices only and purchased items
ttp_df_filtered_view_devices = ttp_df_with_login[
    ttp_df_with_login['view_device'].isin(['mobile', 'desktop']) &
    (ttp_df_with_login['event_observed'] == 1)
]
# TTP Distribution by Device Category (View Device)
plt.figure(figsize=(10, 6))
sns.boxplot(x='view_device', y='days_to_purchase', data=ttp_df_filtered_view_devices, palette='viridis')
plt.title('Time to Purchase Distribution by Device Category (View Device - Purchased Items Only)')
plt.xlabel('Device Category at First View')
plt.ylabel('Time to Purchase (Days)')
plt.yscale('log') # Use log scale due to skewed TTP distribution
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

print("\n--- TTP by Device (Purchase) ---")
# Filter for mobile and desktop devices only for purchase_device and only purchased items
ttp_df_filtered_purchase_devices = ttp_df_with_login[
    ttp_df_with_login['purchase_device'].isin(['mobile', 'desktop']) &
    (ttp_df_with_login['event_observed'] == 1)
]

plt.figure(figsize=(10, 6))
sns.boxplot(x='purchase_device', y='days_to_purchase', data=ttp_df_filtered_purchase_devices, palette='cividis') # Using a different palette for distinction
plt.title('Time to Purchase Distribution by Device Category (Purchase Device - Purchased Items Only)')
plt.xlabel('Device Category at Purchase')
plt.ylabel('Time to Purchase (Days)')
plt.yscale('log')
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

# 2. TTP Distribution by Login Status
plt.figure(figsize=(10, 6))
sns.boxplot(x='login_status', y='days_to_purchase', data=ttp_df_with_login, palette='plasma')
plt.title('Time to Purchase Distribution by Login Status (Purchased Items Only)')
plt.xlabel('Login Status')
plt.ylabel('Time to Purchase (Days)')
plt.yscale('log') # Use log scale due to skewed TTP distribution
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

# 3. TTP Distribution by Returning User Status
print("\n--- TTP by Returning User Status ---")
plt.figure(figsize=(10, 6))
sns.boxplot(x='returning_visitor', y='days_to_purchase', data=ttp_df_with_login, palette='cividis')
plt.title('Time to Purchase Distribution by Returning User Status (Purchased Items Only)')
plt.xlabel('Returning User (0=New, 1=Returning)')
plt.ylabel('Time to Purchase (Days)')
plt.yscale('log') # Use log scale due to skewed TTP distribution
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

# 4. TTP Distribution by Price Tier
print("\n--- TTP by Price Tier ---")
plt.figure(figsize=(10, 6))
sns.boxplot(x='price_tier', y='days_to_purchase', data=ttp_df_with_login, palette='magma', order=['low', 'medium', 'high'])
plt.title('Time to Purchase Distribution by Price Tier (Purchased Items Only)')
plt.xlabel('Price Tier')
plt.ylabel('Time to Purchase (Days)')
plt.yscale('log') # Use log scale due to skewed TTP distribution
plt.grid(True, which="both", ls="-", alpha=0.6)
plt.show()

 **Analysis of Time to Purchase (TTP) Distribution and Related Visualizations**

 **1. TTP Distribution (days_to_purchase)**
The `days_to_purchase` variable exhibits a highly right-skewed distribution. The mean TTP is approximately **1.57 days**, while the median TTP is significantly lower. This disparity highlights that a large majority of purchases occur very quickly after the first view, while a smaller number of transactions take much longer, with the maximum TTP extending to **39 days**.

Breaking down the purchases into time buckets further clarifies this:
*   **< 1 Hour**: 59.8% of purchases
*   **1-24 Hours**: 17.9% of purchases
*   **1-7 Days**: 15.8% of purchases
*   **7+ Days**: 6.5% of purchases

This indicates that approximately **78% of purchases are completed within a single day**, suggesting a strong tendency for immediate or same-day conversions for the observed items.

**Histograms and Box Plots of 'minutes_to_purchase'**: These visualizations clearly confirm the heavily right-skewed nature of the TTP data. The histogram shows a sharp, tall peak very close to zero, representing the numerous quick conversions, followed by a long, flat tail extending towards higher day values, indicating fewer, longer purchase cycles.

While these figures suggest that approximately 78% of purchases are completed within a single day, this data is **not 100% reliable** due to limitations in tracking the longitudinal user journey. Our analysis relies on the user_pseudo_id to link interactions across sessions. Because this **ID is cookie-based**, it cannot track users who switch devices or clear their browser history, which likely causes us to underestimate the true length of the consideration phase for luxury goods.However, we have chosen to proceed with the user_pseudo_id as our primary identifier because the login status metadata reveals that clients are authenticated in only 11.5% of unique sessions. Relying exclusively on logged-in data would result in a **significant loss** of sample size and provide an incomplete view of the broader website visitor activity. Therefore, these TTP metrics should be viewed as the "final mile" of the purchase journey rather than the total time spent in consideration.

**2. Visual Inspection of TTP (Insights from Graphs)**

*   **Log(TTP) vs. Log(Engagement Time) Scatter Plot**: The scatter plot, along with a correlation of approximately **0.06**, indicates a very weak positive relationship between `engagement_time_msec` and `days_to_purchase`. This suggests that while there's a slight tendency for longer engagement to be associated with longer purchase times, the relationship is not strong.

*   **TTP by Device (View and Purchase)**: The box plots comparing TTP by device category (both at the time of first view and at purchase) reveal interesting patterns. Mobile and Desktop have similar distribution in the first view and purhcase events with similar median values. However, `Desktop` device exhibit a more concentrated distribution, indicating that purchases on desktop are more consistently completed within a narrower timeframe compared to mobile, suggesting less variability in TTP for desktop users.

*   **TTP by Login Status**: The box plot clearly shows that `Logged-in` users (labeled '1') tend to have a shorter Time to Purchase and a less dispersed distribution compared to `Not Logged-in` users (labeled '0'), indicating faster and more consistent conversions when a user is logged in at the time of the view event.

*   **TTP by Returning Visitor**
-The distribution shows that returning users (1) tend to have a longer time to purchase compared to new users (0). The median time to purchase is higher for returning users, suggesting slower conversion despite prior engagement.
New users appear to convert more quickly, with a more concentrated distribution at lower time values and while returning users are more experienced, they do not necessarily purchase faster than new users.

*   **TTP by Price Tier**
-The graph shows a clear relationship between price level and time to purchase, with higher-priced items generally requiring longer consideration periods.  
  - Low-priced items tend to convert quickly, as reflected by the lower median and more concentrated distribution at shorter time intervals, suggesting more impulsive or low-risk decisions.  
  - As we move to medium and especially high price tiers, both the median and the spread increase, indicating that users take more time and exhibit more variability before purchasing. The wider interquartile range and presence of many outliers in higher tiers highlight that some purchases can be significantly delayed, likely due to higher involvement and evaluation effort.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# VISUALIZATION (3 BOXPLOTS)
# ------------------------------------------------------------
sections = ['Target', 'Typology', 'Product']
palette_list = ['Set2', 'viridis', 'magma']

# We use ttp_df which contains the processed time-to-purchase data
# Filter for purchased items only to analyze distribution
df_purchased = ttp_df[ttp_df['event_observed'] == 1].copy()
total_transactions_all = len(df_purchased)

for i, section in enumerate(sections):
    # Filter out n/a and unclassified for a professional report output
    df_plot_filtered = df_purchased[~df_purchased[section].isin(['N/a', 'Unclassified', 'n/a', 'Unclassified'])].copy()

    # Calculate value counts for all valid categories in this section BEFORE taking top 10
    value_counts_all = df_plot_filtered[section].value_counts()

    # Select Top 10 to keep the charts readable for the Versace consultants
    top_values = value_counts_all.head(10).index
    df_plot = df_plot_filtered[df_plot_filtered[section].isin(top_values)]

    # Sort by median days to purchase to show conversion speed patterns
    order = df_plot.groupby(section)['days_to_purchase'].median().sort_values().index

    plt.figure(figsize=(12, 6))
    ax = sns.boxplot(
        x=section,
        y='days_to_purchase',
        data=df_plot,
        order=order,
        palette=palette_list[i]
    )

    plt.yscale('log')
    plt.title(f'Time to Purchase Distribution by {section} (with % of total transactions)', fontsize=15)
    plt.xlabel(section, fontsize=12)
    plt.ylabel('Days to Purchase (Log Scale)', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3, linestyle='--')

    # Adjust y-axis limits to make space for annotations
    current_ymin, current_ymax = ax.get_ylim()
    adjusted_current_ymax = max(current_ymax, 1e-3)
    ax.set_ylim(current_ymin, adjusted_current_ymax * 3.0)

    # Add percentage of transactions for each group
    current_xticks = ax.get_xticks()
    text_y_position = adjusted_current_ymax * 1.5

    for k, group_name in enumerate(order):
        transactions_in_group = value_counts_all.get(group_name, 0)
        percentage = (transactions_in_group / total_transactions_all) * 100 if total_transactions_all > 0 else 0

        ax.text(
            current_xticks[k],
            text_y_position,
            f'{percentage:.1f}%',
            color='black',
            ha='center',
            va='bottom',
            fontsize=9,
            bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2')
        )

    plt.tight_layout()
    plt.show()

**Category analysis - TTP**

- **Time to Purchase by Target**:  
  The distribution shows noticeable differences across target segments, with some groups (e.g., Women and Children) exhibiting higher median times to purchase compared to others like Men. Importantly, Men account for the largest share of transactions (46.9%), followed by Women (30.6%), while Home and Lifestyle (12.7%) and Children (9.7%) are smaller segments. This suggests that although Men convert faster and drive most volume, other segments may involve longer and more deliberate decision processes. At the same time, all groups display substantial dispersion and outliers, indicating heterogeneous behavior within each segment.

- **Time to Purchase by Typology**:  
  When looking at product typologies, we observe a clear gradient in purchase timing. Categories such as Underwear and beachwear (29.3% of transactions) dominate in volume and show relatively short time-to-purchase, suggesting more impulsive or habitual buying behavior. In contrast, categories like Accessories (16.9%) and Clothing (8.8%) have longer and more dispersed purchase times, indicating higher involvement. Smaller categories (e.g., Baby 0–18m at 3.4%) contribute less to total transactions but still exhibit meaningful variability in timing.

- **Time to Purchase by Product**:  
  At a more granular product level, differences become even more pronounced. High-frequency items such as swimsuits (7.5%) and briefs/boxers (around 5–6%) tend to convert faster, reflecting lower consideration effort. Conversely, products like sneakers, t-shirts and polos, or swimwear (lower individual shares, around 2–5%) show longer and more variable time-to-purchase, suggesting more complex decision-making. Overall, combining timing with transaction shares highlights that faster-converting products also tend to drive a significant portion of volume, while slower ones contribute less but involve deeper consideration.

### **Comparisons of TTP and Conversion**

This section compares conversion rates and purchase timing across different customer and session characteristics, such as device, login status, traffic source, search activity, and add-to-cart behavior. The goal is to identify the main observed drivers of faster conversion.

In [ ]:
def driver_summary(data, group_col):
    out = (
        data.groupby(group_col)
        .agg(
            episodes=("item_id", "count"),
            purchases=("event_observed", "sum"),
            conversion_rate=("event_observed", "mean"),
            median_duration_all=("duration_days", "median")
        )
        .reset_index()
    )

    purchasers_only = (
        data[data["event_observed"] == 1]
        .groupby(group_col)["duration_days"]
        .median()
        .rename("median_duration_purchasers")
        .reset_index()
    )

    out = out.merge(purchasers_only, on=group_col, how="left")
    out["conversion_rate"] = (out["conversion_rate"] * 100).round(2)
    return out


In [ ]:
ttp_df.columns

In [ ]:
#device
device_table = driver_summary(ttp_df, "view_device")
print(device_table)

In [ ]:
#login
login_table = driver_summary(ttp_df, "logged_in_first_view")
print(login_table)

In [ ]:
#add to cart in same session
cart_table = driver_summary(ttp_df, "add_to_cart_same_session")
print(cart_table)

In [ ]:
#price_tier
price_tier_table = driver_summary(ttp_df, "price_tier")
print(price_tier_table)

In [ ]:
#returning_visitor
returning_visitor_table = driver_summary(ttp_df, "returning_visitor")
print(returning_visitor_table)

The summaries suggest a consistent pattern: variables associated with **higher purchase intent** are also linked to **higher conversion** and **shorter time to purchase**.

- **Device.** Mobile shows the highest conversion rate and a shorter median duration than desktop, suggesting that mobile users in this dataset are not only browsing but also converting relatively quickly.
- **Login status.** Users already logged in at first view convert more often and faster, which is consistent with stronger prior intent or familiarity with the brand.
- **Price tier.** Lower-price items convert more often and faster, while higher-price items show lower conversion and longer consideration, which is plausible in a luxury e-commerce setting.
- **Returning Visitor.** Returning users have a lower conversion rate compared to new users. However, returning users also have a longer median duration for all episodes and for purchasers only compared to new users. This suggests that while new users convert more readily and quickly, returning users may have a more extended consideration phase or engage with the platform in different ways before purchasing.

Overall, the results are coherent with a luxury purchase journey: some users convert almost immediately, while others take longer, especially when intent is weaker or the product is more expensive.

##**3. UTM Analysis**

We first extracted campaign-related information from the URL by parsing page_location and identifying standard tracking parameters such as utm_source, utm_medium, utm_campaign, and campaign_name. From this, we created a **has_utm** indicator to flag sessions that likely originated from a **paid or campaign-driven traffic source**.

Next, we moved from the event level to the session level, aggregating each user’s activity by session and identifying the first purchase session in the journey. We then restricted the analysis to the sequence of sessions observed up to the first purchase, so that campaign features were constructed only from the pre-conversion path.

Using this pre-purchase journey, we built user-level variables describing the role of paid traffic in the path to first purchase. In particular, we measured whether the user was ever exposed to paid traffic before converting (paid_entry), when the first paid touch occurred (first_paid_session_rank), how many paid sessions appeared before conversion, and whether the purchase itself happened in a paid session (purchase_from_paid_last_touch).

Finally, we merged these campaign-journey features into the survival dataset (ttp_df) so that each user-item observation also carries information about the user’s broader acquisition path. The business objective of this step is to understand whether paid traffic seems to accelerate first purchase, whether it acts mainly as an acquisition or remarketing channel, and whether its contribution is primarily assistive or conversion-closing.

In [ ]:
from urllib.parse import urlparse, parse_qs

def parse_url_features(url):
    if pd.isna(url):
        return {
            "has_utm": 0,
            "utm_source": None,
            "utm_medium": None,
            "utm_campaign": None,
            "campaign_name": None
        }
    try:
        q = parse_qs(urlparse(str(url)).query)
        utm_source = q.get("utm_source", [None])[0]
        utm_medium = q.get("utm_medium", [None])[0]
        utm_campaign = q.get("utm_campaign", [None])[0]
        campaign_name = q.get("campaign", [None])[0]

        return {
            "has_utm": int(any([utm_source, utm_medium, utm_campaign, campaign_name])),
            "utm_source": utm_source,
            "utm_medium": utm_medium,
            "utm_campaign": utm_campaign,
            "campaign_name": campaign_name
        }
    except Exception:
        return {
            "has_utm": 0,
            "utm_source": None,
            "utm_medium": None,
            "utm_campaign": None,
            "campaign_name": None
        }

if "page_location" in df.columns and "has_utm" not in df.columns:
    url_features = df["page_location"].apply(parse_url_features).apply(pd.Series)
    df = pd.concat([df, url_features], axis=1)
elif "has_utm" not in df.columns:
    df["has_utm"] = 0
    df["utm_source"] = None
    df["utm_medium"] = None
    df["utm_campaign"] = None
    df["campaign_name"] = None

In [ ]:
df

In [ ]:
#session level campaign features before first purchase
session_id_col = "session_id" if "session_id" in df.columns else "ga_session_id"

session_agg_dict = {
    "event_datetime": "min",
    "has_utm": "max",
    "utm_source": "first",
    "utm_medium": "first",
    "utm_campaign": "first",
    "campaign_name": "first"
}

if "session_rank" in df.columns:
    session_agg_dict["session_rank"] = "first"

session_df = (
    df.groupby(["user_pseudo_id", session_id_col], as_index=False)
    .agg(session_agg_dict)
    .rename(columns={
        "event_datetime": "session_start",
        session_id_col: "session_id"
    })
)

# session_rank fallback
if "session_rank" not in session_df.columns:
    session_df = session_df.sort_values(["user_pseudo_id", "session_start"]).copy()
    session_df["session_rank"] = session_df.groupby("user_pseudo_id").cumcount() + 1

# flag purchase session
purchase_sessions = (
    df.loc[df["event_name"] == "purchase", ["user_pseudo_id", session_id_col]]
    .drop_duplicates()
    .rename(columns={session_id_col: "session_id"})
)

purchase_sessions["is_purchase_session"] = 1

session_df = session_df.merge(
    purchase_sessions,
    on=["user_pseudo_id", "session_id"],
    how="left"
)

session_df["is_purchase_session"] = session_df["is_purchase_session"].fillna(0).astype(int)

# first buying session
first_purchase_session = (
    session_df.loc[session_df["is_purchase_session"] == 1]
    .sort_values(["user_pseudo_id", "session_rank", "session_start"])
    .drop_duplicates("user_pseudo_id", keep="first")
    .rename(columns={
        "session_id": "first_purchase_session_id",
        "session_rank": "first_purchase_session_rank",
        "session_start": "first_purchase_session_time"
    })[[
        "user_pseudo_id",
        "first_purchase_session_id",
        "first_purchase_session_rank",
        "first_purchase_session_time"
    ]]
)


user_sessions_pre_first_purchase = session_df.merge(
    first_purchase_session,
    on="user_pseudo_id",
    how="left"
)


user_sessions_pre_first_purchase = user_sessions_pre_first_purchase.loc[
    user_sessions_pre_first_purchase["first_purchase_session_rank"].isna() |
    (user_sessions_pre_first_purchase["session_rank"] <= user_sessions_pre_first_purchase["first_purchase_session_rank"])
].copy()

In [ ]:
#user_level paid / campaign features
def build_paid_journey_features(g):
    g = g.sort_values(["session_rank", "session_start"]).copy()

    paid_sessions = g.loc[g["has_utm"] == 1].copy()

    paid_entry = int(len(paid_sessions) > 0)

    first_paid_session_rank = (
        paid_sessions["session_rank"].iloc[0] if len(paid_sessions) > 0 else np.nan
    )

    first_paid_session_id = (
        paid_sessions["session_id"].iloc[0] if len(paid_sessions) > 0 else np.nan
    )

    first_paid_utm_campaign = (
        paid_sessions["utm_campaign"].iloc[0] if len(paid_sessions) > 0 else np.nan
    )

    first_paid_utm_source = (
        paid_sessions["utm_source"].iloc[0] if len(paid_sessions) > 0 else np.nan
    )

    first_paid_utm_medium = (
        paid_sessions["utm_medium"].iloc[0] if len(paid_sessions) > 0 else np.nan
    )

    n_paid_sessions_before_first_purchase = len(paid_sessions)


    if g["first_purchase_session_rank"].notna().any():
        purchase_rank = g["first_purchase_session_rank"].iloc[0]
        purchase_session = g.loc[g["session_rank"] == purchase_rank].copy()

        if len(purchase_session) > 0:
            purchase_from_paid_last_touch = int(purchase_session["has_utm"].iloc[0] == 1)
            purchase_utm_campaign = purchase_session["utm_campaign"].iloc[0]
            purchase_utm_source = purchase_session["utm_source"].iloc[0]
            purchase_utm_medium = purchase_session["utm_medium"].iloc[0]
        else:
            purchase_from_paid_last_touch = np.nan
            purchase_utm_campaign = np.nan
            purchase_utm_source = np.nan
            purchase_utm_medium = np.nan
    else:
        purchase_rank = np.nan
        purchase_from_paid_last_touch = np.nan
        purchase_utm_campaign = np.nan
        purchase_utm_source = np.nan
        purchase_utm_medium = np.nan


    paid_touch_session_gap = (
        purchase_rank - first_paid_session_rank
        if pd.notna(purchase_rank) and pd.notna(first_paid_session_rank)
        else np.nan
    )

    # paid first touch
    paid_first_touch = int(
        pd.notna(first_paid_session_rank) and first_paid_session_rank == 1
    )

    # paid late
    paid_late_entry = int(
        pd.notna(first_paid_session_rank) and first_paid_session_rank >= 3
    )

    return pd.Series({
        "paid_entry": paid_entry,
        "first_paid_session_rank": first_paid_session_rank,
        "first_paid_session_id": first_paid_session_id,
        "first_paid_utm_campaign": first_paid_utm_campaign,
        "first_paid_utm_source": first_paid_utm_source,
        "first_paid_utm_medium": first_paid_utm_medium,
        "n_paid_sessions_before_first_purchase": n_paid_sessions_before_first_purchase,
        "purchase_from_paid_last_touch": purchase_from_paid_last_touch,
        "purchase_utm_campaign": purchase_utm_campaign,
        "purchase_utm_source": purchase_utm_source,
        "purchase_utm_medium": purchase_utm_medium,
        "paid_touch_session_gap": paid_touch_session_gap,
        "paid_first_touch": paid_first_touch,
        "paid_late_entry": paid_late_entry
    })

paid_user_features = (
    user_sessions_pre_first_purchase
    .groupby("user_pseudo_id", as_index=False)
    .apply(build_paid_journey_features)
    .reset_index(drop=True)
)

paid_user_features.head()

In [ ]:
import numpy as np
import pandas as pd

# 1. Define the journey feature builder
def build_paid_journey_features(g):
    g = g.sort_values(['session_rank', 'session_start']).copy()
    paid_sessions = g.loc[g['has_utm'] == 1].copy()
    paid_entry = int(len(paid_sessions) > 0)
    first_paid_session_rank = paid_sessions['session_rank'].iloc[0] if len(paid_sessions) > 0 else np.nan
    first_paid_session_id = paid_sessions['session_id'].iloc[0] if len(paid_sessions) > 0 else np.nan
    first_paid_utm_campaign = paid_sessions['utm_campaign'].iloc[0] if len(paid_sessions) > 0 else np.nan
    first_paid_utm_source = paid_sessions['utm_source'].iloc[0] if len(paid_sessions) > 0 else np.nan
    first_paid_utm_medium = paid_sessions['utm_medium'].iloc[0] if len(paid_sessions) > 0 else np.nan
    n_paid_sessions_before_first_purchase = len(paid_sessions)

    if g['first_purchase_session_rank'].notna().any():
        purchase_rank = g['first_purchase_session_rank'].iloc[0]
        purchase_session = g.loc[g['session_rank'] == purchase_rank].copy()
        if len(purchase_session) > 0:
            purchase_from_paid_last_touch = int(purchase_session['has_utm'].iloc[0] == 1)
            purchase_utm_campaign = purchase_session['utm_campaign'].iloc[0]
            purchase_utm_source = purchase_session['utm_source'].iloc[0]
            purchase_utm_medium = purchase_session['utm_medium'].iloc[0]
        else:
            purchase_from_paid_last_touch, purchase_utm_campaign, purchase_utm_source, purchase_utm_medium = np.nan, np.nan, np.nan, np.nan
    else:
        purchase_rank, purchase_from_paid_last_touch, purchase_utm_campaign, purchase_utm_source, purchase_utm_medium = np.nan, np.nan, np.nan, np.nan, np.nan

    paid_touch_session_gap = purchase_rank - first_paid_session_rank if pd.notna(purchase_rank) and pd.notna(first_paid_session_rank) else np.nan
    paid_first_touch = int(pd.notna(first_paid_session_rank) and first_paid_session_rank == 1)
    paid_late_entry = int(pd.notna(first_paid_session_rank) and first_paid_session_rank >= 3)

    return pd.Series({
        'paid_entry': paid_entry,
        'first_paid_session_rank': first_paid_session_rank,
        'first_paid_session_id': first_paid_session_id,
        'first_paid_utm_campaign': first_paid_utm_campaign,
        'first_paid_utm_source': first_paid_utm_source,
        'first_paid_utm_medium': first_paid_utm_medium,
        'n_paid_sessions_before_first_purchase': n_paid_sessions_before_first_purchase,
        'purchase_from_paid_last_touch': purchase_from_paid_last_touch,
        'purchase_utm_campaign': purchase_utm_campaign,
        'purchase_utm_source': purchase_utm_source,
        'purchase_utm_medium': purchase_utm_medium,
        'paid_touch_session_gap': paid_touch_session_gap,
        'paid_first_touch': paid_first_touch,
        'paid_late_entry': paid_late_entry
    })

# 2. Execute user-level aggregation
paid_user_features = user_sessions_pre_first_purchase.groupby('user_pseudo_id', as_index=False).apply(build_paid_journey_features).reset_index(drop=True)

# 3. Merge into ttp_df
paid_cols_to_merge = [col for col in paid_user_features.columns if col != 'user_pseudo_id']
ttp_df = ttp_df.drop(columns=[c for c in paid_cols_to_merge if c in ttp_df.columns], errors='ignore')
ttp_df = ttp_df.merge(paid_user_features, on='user_pseudo_id', how='left')

# 4. Construct user_paid_df for timing analysis
user_paid_df = ttp_df[['user_pseudo_id', 'paid_entry', 'first_paid_session_rank', 'n_paid_sessions_before_first_purchase', 'purchase_from_paid_last_touch', 'first_paid_utm_campaign', 'paid_touch_session_gap', 'first_paid_utm_source', 'first_paid_utm_medium']].drop_duplicates('user_pseudo_id').copy()

first_touch_user = df.groupby('user_pseudo_id', as_index=False).agg(first_touch_time=('event_datetime', 'min'))
first_purchase_user = df.loc[df['event_name'] == 'purchase'].groupby('user_pseudo_id', as_index=False).agg(first_purchase_time=('event_datetime', 'min'))

user_paid_df = user_paid_df.merge(first_touch_user, on='user_pseudo_id', how='left')
user_paid_df = user_paid_df.merge(first_purchase_user, on='user_pseudo_id', how='left')
user_paid_df['days_to_first_purchase_user'] = (user_paid_df['first_purchase_time'] - user_paid_df['first_touch_time']).dt.total_seconds() / (3600 * 24)

print('Paid entry features and user-level journey metrics integrated successfully.')
display(user_paid_df.head())

In [ ]:
ttp_df[[
    "user_pseudo_id",
    "item_id",
    "event_observed",
    "paid_entry",
    "first_paid_session_rank",
    "n_paid_sessions_before_first_purchase",
    "purchase_from_paid_last_touch",
    "first_paid_utm_campaign",
    "paid_touch_session_gap"
]].head(10)

In [ ]:
user_paid_df = (
    ttp_df[
        [
            "user_pseudo_id",
            "paid_entry",
            "first_paid_session_rank",
            "n_paid_sessions_before_first_purchase",
            "purchase_from_paid_last_touch",
            "first_paid_utm_campaign",
            "paid_touch_session_gap",
            "first_paid_utm_source",
            "first_paid_utm_medium"
        ]
    ]
    .drop_duplicates("user_pseudo_id")
    .copy()
)

In [ ]:
first_touch_user = (
    df.groupby("user_pseudo_id", as_index=False)
    .agg(first_touch_time=("event_datetime", "min"))
)

first_purchase_user = (
    df.loc[df["event_name"] == "purchase"]
    .groupby("user_pseudo_id", as_index=False)
    .agg(first_purchase_time=("event_datetime", "min"))
)

user_paid_df = user_paid_df.merge(first_touch_user, on="user_pseudo_id", how="left")
user_paid_df = user_paid_df.merge(first_purchase_user, on="user_pseudo_id", how="left")

user_paid_df["days_to_first_purchase_user"] = (
    (user_paid_df["first_purchase_time"] - user_paid_df["first_touch_time"]).dt.total_seconds() / (3600 * 24)
)

user_paid_df.head()

In [ ]:
paid_entry_summary = (
    user_paid_df.groupby("paid_entry", as_index=False)
    .agg(
        n_users=("user_pseudo_id", "nunique"),
        avg_days_to_first_purchase=("days_to_first_purchase_user", "mean"),
        median_days_to_first_purchase=("days_to_first_purchase_user", "median")
    )
)

paid_entry_summary

In [ ]:
def classify_paid_timing(x):
    if pd.isna(x):
      return "no_paid"
    elif x == 1:
        return "paid_first_touch"
    elif x <= 3:
        return "paid_early"
    else:
        return "paid_late"

user_paid_df["paid_entry_timing"] = user_paid_df["first_paid_session_rank"].apply(classify_paid_timing)
timing_bucket_summary = (
    user_paid_df.groupby("paid_entry_timing", as_index=False)
    .agg(
        n_users=("user_pseudo_id", "nunique"),
        avg_days_to_first_purchase=("days_to_first_purchase_user", "mean"),
        median_days_to_first_purchase=("days_to_first_purchase_user", "median")
    )
)

timing_bucket_summary

In [ ]:
comparison_table = pd.crosstab(
    user_paid_df["paid_entry"],
    user_paid_df["purchase_from_paid_last_touch"],
    normalize="index"
)

comparison_table

In [ ]:
last_touch_summary = (
    user_paid_df.groupby("purchase_from_paid_last_touch", as_index=False)
    .agg(
        n_users=("user_pseudo_id", "nunique"),
        avg_days_to_first_purchase=("days_to_first_purchase_user", "mean"),
        median_days_to_first_purchase=("days_to_first_purchase_user", "median")
    )

)

last_touch_summary

In [ ]:
source_medium_summary = (
    user_paid_df.loc[user_paid_df["paid_entry"] == 1]
    .groupby(["first_paid_utm_source", "first_paid_utm_medium"], as_index=False)
    .agg(
        n_users=("user_pseudo_id", "nunique"),
        avg_days_to_first_purchase=("days_to_first_purchase_user", "mean"),
        median_days_to_first_purchase=("days_to_first_purchase_user", "median")
    )
    .sort_values("n_users", ascending=False)
)

source_medium_summary.head(20)

In [ ]:
campaign_summary = (
    user_paid_df.loc[user_paid_df["paid_entry"] == 1]
    .groupby("first_paid_utm_campaign", as_index=False)
    .agg(
        n_users=("user_pseudo_id", "nunique"),
        avg_days_to_first_purchase=("days_to_first_purchase_user", "mean"),
        median_days_to_first_purchase=("days_to_first_purchase_user", "median")
    )
)

campaign_summary = campaign_summary.loc[campaign_summary["n_users"] >= 10]
campaign_summary.sort_values("avg_days_to_first_purchase").head(15)

**Results and insights**

The comparison between users with and without `paid_entry` suggests that paid traffic is not simply associated with faster conversion. Users exposed to at least one campaign-driven session before their first purchase show both a higher average and a higher median time to first purchase than users with no paid entry. From a business perspective, this indicates that paid traffic often supports longer consideration journeys rather than only driving immediate conversion. In other words, paid channels may be helping Versace stay present during the evaluation phase, especially for users who require more time before purchasing.

The timing analysis adds a more strategic layer to this result. When paid traffic appears as the **first touchpoint**, the journey is shorter than when paid enters later. This suggests that paid can still play an **acquisition role**, especially when it is the user’s first point of entry into the brand’s digital journey. However, when the first paid touch appears only after the first few sessions, the path to purchase becomes much longer. This points to a second function of paid traffic: **remarketing and reactivation**. Business-wise, this means that paid campaigns should not all be treated the same way, because some are attracting new high-intent visitors, while others are re-engaging users already in a long decision process.

The relationship between `paid_entry` and `purchase_from_paid_last_touch` shows that paid traffic is not only an assisting channel. A large share of users with paid entry also complete their first purchase in a paid session, meaning that campaigns often remain relevant until the final conversion step. This suggests that paid traffic contributes both to **consideration** and to **conversion closing**. For Versace, this is important because it implies that evaluating paid channels only through simple last-click logic may miss part of their value: some campaigns initiate or support the journey, while others also help close the sale.

The source/medium results show strong heterogeneity across paid channels. Search- and shopping-led traffic, such as Google Shopping or affiliate-driven entries, is associated with much shorter times to first purchase, which is consistent with **high-intent traffic capture**. In contrast, email- and social-driven entries are associated with longer journeys, suggesting a stronger role in **inspiration, nurturing, and re-engagement**. SEM appears to sit between these two extremes, likely combining users with strong purchase intent and users still exploring the brand. From a business standpoint, this means that paid channels should be managed with **different success metrics depending on their role in the funnel**.

The campaign-level analysis reinforces the same conclusion. Some campaigns, especially search- and shopping-oriented ones, seem to attract users who are nearly ready to buy, while others, such as dynamic social campaigns, appear to operate within longer and more gradual purchase journeys. This implies that budget allocation should not be based only on final conversion volume, but also on the **type of journey each campaign is generating**. Campaigns that bring fast-converting users may be optimized for direct response, while campaigns associated with longer journeys may be better evaluated as tools for nurturing and funnel progression.

Overall, the evidence suggests that paid traffic plays **multiple roles** in the Versace customer journey. Some paid channels act as acquisition tools, some as nurturing mechanisms, and some as closing touchpoints. The key managerial implication is that paid campaigns should not be evaluated under a single attribution logic. Instead, Versace should distinguish between:
- channels that capture **high-intent demand**,
- channels that support **mid-funnel consideration**,
- and channels that help **reactivate and convert** users later in the journey.

First, Versace should separate **acquisition campaigns** from **nurturing/remarketing campaigns** in both media strategy and performance evaluation. Second, the company should move beyond a purely **last-click view** of performance, because some paid touchpoints appear to create value earlier in the journey. Third, landing pages, product page experiences, and creative messages should be aligned with the role of the channel: conversion-oriented channels should emphasize strong product visibility and frictionless purchase, while social and email channels should support storytelling, reminder mechanisms, and continuity across sessions.

## **4. Clustering**

Before running the Survival Analysis we cluster the users in three buckets
using K-means, after evaluating different values of k through the silhouette score and the elbow method. We chose the 3-cluster solution as it allows for a more nuanced and actionable segmentation of user behavior.

We clustered users based solely on purely behavioral features, to ensure that the segments capture actual browsing dynamics and intent levels, avoiding potential biases or 'leakage' from static demographic or categorical variables.

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

#prepare df_cluster

# Aggregate features at the user level (user_pseudo_id)
df_cluster = ttp_df.groupby("user_pseudo_id").agg({
    #"ga_session_number_at_first_visit": "first",
    "view_item_price": "mean",
    "n_distinct_other_items_viewed": "mean",
    "n_views_item": "mean"
}).rename(columns={
    "view_item_price": "avg_view_item_price",
    "n_distinct_other_items_viewed": "avg_n_distinct_other_items_viewed",
    "n_views_item": "avg_n_views_item"
}).reset_index()


#select pure behavioral features
pure_features = [
   # "ga_session_number_at_first_visit",
    "avg_view_item_price",
    "avg_n_distinct_other_items_viewed",
    "avg_n_views_item"
]

# Clean copy
df_kmeans = df_cluster.dropna(subset=pure_features).copy()

#log-transform & robust scaling

log_features = np.log1p(df_kmeans[pure_features])

# Using RobustScaler as requested to handle outliers
scaler = RobustScaler()
scaled_features = scaler.fit_transform(log_features)


#elbow method + silhouette score
k_range = range(2, 9)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(scaled_features)

    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(scaled_features, labels))

# Print scores
scores_df = pd.DataFrame({
    "k": list(k_range),
    "inertia": inertias,
    "silhouette_score": sil_scores
})

print("Elbow and Silhouette results (with RobustScaler and n_views_item):")
print(scores_df)

# Plot elbow + silhouette side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_range), sil_scores, marker="o")
axes[1].set_title("Silhouette Scores")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()


#final model, k=3
kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=20)
df_kmeans["cluster"] = kmeans_3.fit_predict(scaled_features)

final_silhouette = silhouette_score(scaled_features, df_kmeans["cluster"])
print(f"\nFinal silhouette score for K=3: {final_silhouette:.3f}")


#cluster sizes
cluster_sizes = df_kmeans["cluster"].value_counts().sort_index()
print("\nCluster sizes for K=3:")
print(cluster_sizes)


#business profile heatmap
real_means = df_kmeans.groupby("cluster")[pure_features].mean()

# Standardize for heatmap coloring to see relative levels
heatmap_colors = pd.DataFrame(
    StandardScaler().fit_transform(real_means),
    index=real_means.index,
    columns=real_means.columns
)

real_means_display = real_means.copy().round(2)
heatmap_colors.index = [f"Cluster {i}" for i in heatmap_colors.index]
real_means_display.index = [f"Cluster {i}" for i in real_means_display.index]

plt.figure(figsize=(10, 5))
sns.heatmap(
    heatmap_colors,
    cmap="coolwarm",
    annot=real_means_display,
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "Standardized relative level"}
)
plt.title("Behavioral Profile of Clusters (K=3, RobustScaler)")
plt.xlabel("Behavioral Features")
plt.ylabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("\n--- Cluster-wise Metrics ---")
# 1. Aggregate cluster metrics using the numeric cluster ID and item_revenue
# We join ttp_df with df_kmeans to ensure the 'cluster' column is available
cluster_metrics = ttp_df.merge(df_kmeans[['user_pseudo_id', 'cluster']], on='user_pseudo_id', how='left').groupby('cluster').agg(
    mean_days_to_purchase=('days_to_purchase', 'mean'),
    num_items=('item_id', 'count'),
    total_revenue_volume=('item_revenue', 'sum'),
    num_users=('user_pseudo_id', 'nunique'),
    num_transactions=('transaction_id', 'nunique')
).reset_index()

# 2. Calculate Derived Metrics
cluster_metrics['mean_transaction_value'] = (cluster_metrics['total_revenue_volume'] / cluster_metrics['num_transactions']).round(2)
cluster_metrics['mean_transactions_per_person'] = (cluster_metrics['num_transactions'] / cluster_metrics['num_users']).round(2)
cluster_metrics['mean_spend_per_person'] = (cluster_metrics['total_revenue_volume'] / cluster_metrics['num_users']).round(2)

# 3. Formatting
cluster_metrics['mean_days_to_purchase'] = cluster_metrics['mean_days_to_purchase'].round(2)
cluster_metrics['total_revenue_volume'] = cluster_metrics['total_revenue_volume'].round(2)

# Display results sorted by the K-means cluster number
display_cols = [
    'cluster', 'num_users', 'num_transactions', 'total_revenue_volume',
    'mean_transaction_value', 'mean_transactions_per_person', 'mean_spend_per_person', 'mean_days_to_purchase'
]
print(cluster_metrics[display_cols].sort_values('cluster'))

In [ ]:
cluster_name_mapping = {
    0: 'A',
    1: 'B',
    2: 'C'
}

df_kmeans['cluster_name'] = df_kmeans['cluster'].map(cluster_name_mapping)

# Map the cluster_name directly to ttp_df based on 'user_pseudo_id'
# This will add a new 'cluster_name' column to ttp_df
# If a 'user_pseudo_id' in ttp_df is not found in df_kmeans, its 'cluster_name' will be NaN.
ttp_df['cluster_name'] = ttp_df['user_pseudo_id'].map(df_kmeans.set_index('user_pseudo_id')['cluster_name'])

print("Cluster sizes with new names:")
print(df_kmeans['cluster_name'].value_counts().sort_index())
print("\nFirst 5 rows of ttp_df with new 'cluster_name' column:")
display(ttp_df[['user_pseudo_id', 'item_id', 'cluster_name']].head())

In [ ]:

sns.set_style("whitegrid")

plt.figure(figsize=(12, 5))


# Scatterplot 1: avg_view_item_price vs avg_n_views_item
plt.subplot(1, 3, 2)
sns.scatterplot(
    x='avg_view_item_price',
    y='avg_n_views_item',
    hue='cluster_name',
    data=df_kmeans,
    palette='viridis',
    s=80, alpha=0.7
)
plt.title('Price vs. Item Re-views')
plt.xlabel('Average Viewed Item Price')
plt.ylabel('Avg. # Item Re-views')

# Scatterplot 2: avg_n_distinct_other_items_viewed vs avg_n_views_item
plt.subplot(1, 3, 3)
sns.scatterplot(
    x='avg_n_distinct_other_items_viewed',
    y='avg_n_views_item',
    hue='cluster_name',
    data=df_kmeans,
    palette='viridis',
    s=80, alpha=0.7
)
plt.title('Distinct Items Viewed vs. Item Re-views')
plt.xlabel('Avg. # Distinct Other Items Viewed')
plt.ylabel('Avg. # Item Re-views')

plt.tight_layout()
plt.show()



Based on the Cluster-wise Metrics and the Behavioral Profile Heatmap, we can segment the user base into three distinct categories based on their spending power, engagement depth, and purchase velocity.

**Cluster 0 - A: High-Value Power Shoppers**
- Monetary Value: This is the "VIP" segment. They generate the highest total revenue ($183,452.50) and have the highest mean spend per person.
- Search Behavior: They target premium products (highest avg_view_item_price) and are highly exploratory, viewing the most distinct items (6.5 on average).
- Efficiency: They have the highest mean_transactions_per_person (1.06), indicating they are reliable converters.

**Cluster 1 - B: Cautious Researchers**
- Monetary Value: A mid-tier segment with a mean spend of $252.90.
- Search Behavior: They exhibit "deep-dive" behavior. While they look at fewer distinct items than Cluster 0, they have the highest repetition rate, viewing the same items 3.31 times on average.
- Purchase Friction: They take the longest to decide, with a mean_days_to_purchase of 2.72.

**Cluster 2 - C: Low-Engagement / Impulse Buyers**
- Monetary Value: This group has the lowest mean spend ($166.15) and targets the most affordable items (lowest avg_view_item_price).
- Search Behavior: Very low engagement metrics across the board; they don't browse much and don't return to items often.
- Purchase Velocity: Surprisingly, they have the fastest conversion time (0.43 days).



## **5. Survival Analysis**


This section employs survival analysis techniques to understand the dynamics of the Time to Purchase (TTP). We use:

*   **Kaplan-Meier Survival Curves:** To visualize the probability of an item not yet being purchased over time, allowing for comparisons across different groups. This section includes an analysis based on sessions
*   **Cox Proportional Hazards Model:** To identify factors that. accelerate or delay conversion, accounting for various user and item characteristics.

### **5a. Kaplan-Meier survival curves & Session Analysis**

We now estimate **Kaplan–Meier survival curves**, which provide a non-parametric way to visualize how the probability of not yet purchasing evolves over time for different groups.

In [ ]:
!pip install lifelines

In [ ]:
# Kaplan-Meier survival curve for Time to Purchase
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

plt.figure(figsize=(9, 5))

kmf.fit(
    durations=ttp_df["duration_days"],
    event_observed=ttp_df["event_observed"],
    label="All user-item pairs"
)

kmf.plot_survival_function(ci_show=True)

plt.title("Kaplan-Meier Survival Curve: Time to Purchase")
plt.xlabel("Days since first item view")
plt.ylabel("Probability item is not yet purchased")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Add  to cart in first Session
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()

plt.figure(figsize=(8,5))
for label, sub in ttp_df.groupby("add_to_cart_same_session"):
    kmf.fit(sub["duration_days"], event_observed=sub["event_observed"], label=f"add_to_cart={label}")
    kmf.plot_survival_function()

plt.title("Survival Curve by Add-to-Cart in First Session")
plt.xlabel("Days since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True)
plt.show()

In [ ]:
#price tier
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
kmf = KaplanMeierFitter()

for label, sub in ttp_df.dropna(subset=["price_tier"]).groupby("price_tier"):
    kmf.fit(sub["duration_days"], event_observed=sub["event_observed"], label=f"price_tier={label}")
    kmf.plot_survival_function(ci_show = True)

plt.title("Survival Curve by Price Tier")
plt.xlabel("Days since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#logged in first view
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
kmf = KaplanMeierFitter()

for label, sub in ttp_df.groupby("logged_in_first_view"):
    kmf.fit(sub["duration_days"], event_observed=sub["event_observed"], label=f"logged_in={label}")
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Login Status at First View")
plt.xlabel("Days since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Ensure paid_entry is mapped from user_paid_df before plotting
if 'paid_entry' not in ttp_df.columns and 'user_paid_df' in globals():
    ttp_df = ttp_df.merge(user_paid_df[['user_pseudo_id', 'paid_entry']], on='user_pseudo_id', how='left')

plt.figure(figsize=(8,5))
kmf = KaplanMeierFitter()

# Filter out potential NaNs for a clean plot
plot_km_df = ttp_df.dropna(subset=['paid_entry'])

for label, sub in plot_km_df.groupby("paid_entry"):
    kmf.fit(sub["duration_days"], event_observed=sub["event_observed"], label=f"paid_entry={label}")
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Paid vs Organic Entry")
plt.xlabel("Days since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

# 1. Initialize the Kaplan-Meier Fitter
kmf = KaplanMeierFitter()

plt.figure(figsize=(8, 5))

# 2. Filter out items without a cluster assignment
plot_km_clusters = ttp_df.dropna(subset=['cluster_name'])

# 3. Plot survival curves for each cluster (A, B, C)
# Sorting the unique clusters to ensure alphabetical order in the legend
for cluster_id in sorted(plot_km_clusters['cluster_name'].unique()):
    sub = plot_km_clusters[plot_km_clusters['cluster_name'] == cluster_id]

    kmf.fit(
        durations=sub["duration_days"],
        event_observed=sub["event_observed"],
        label=f"Cluster {cluster_id}"
    )
    kmf.plot_survival_function(ci_show=True)

# 4. Customizing the Plot
plt.title("Kaplan-Meier Survival Curves by Behavioral Cluster", fontsize=15)
plt.xlabel("Days since first item view", fontsize=12)
plt.ylabel("Probability item is not yet purchased", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title="User Segments")
plt.tight_layout()
plt.show()

**Comments on the Kaplan–Meier curves**

- **Overall survival curve.** The curve drops sharply at the very beginning and then flattens out, showing that a relevant share of purchases happens soon after the first item view. After the initial drop, the decline becomes much slower, suggesting that many remaining user-item pairs stay in a long consideration phase without converting quickly.

- **Add-to-cart in the first session.** The separation is extremely strong: items added to cart immediately have a much steeper decline in the survival function, meaning a much shorter time to purchase. By contrast, items not added to cart in the first session remain overwhelmingly non-purchased throughout the observation window, confirming that immediate carting is one of the strongest signals of high purchase intent.

- **Price tier.** The curves are clearly ordered by price: low-price items decline the fastest, medium-price items are slower, and high-price items remain non-purchased for longer. This indicates that price is a major driver of time to purchase, with more expensive products requiring a longer and more deliberate consideration process.

- **Login status at first view.** Logged-in users show a consistently lower survival probability over time, meaning they tend to purchase faster than non-logged users. This is consistent with the idea that logged-in visitors are more engaged, more familiar with the brand, or already closer to conversion when they first view the item.

- **Paid vs Organic Entry.** Paid traffic exhibits a higher initial engagement but maintains a significantly longer survival tail compared to organic entry. This suggests that while campaigns successfully capture attention, they often attract users in a research-heavy consideration phase. Consequently, paid channels should be evaluated as mid-funnel nurturing tools rather than purely immediate conversion drivers.

- **Behavioral Clusters.** The survival curves starkly visualize the differing purchase velocities among our user segments, with Cluster C (The Impulse Entry-Buyers) dropping most sharply at Day 0, reflecting their immediate, high-velocity conversions. Conversely, Cluster A (The Focused Core Buyers) maintains the highest "survival" probability, demonstrating their prolonged deliberation process where the majority of users do not purchase immediately. Finally, Cluster B (The Aspirational Explorers) falls steadily between the two, illustrating a moderate conversion speed that stabilizes around the two-week mark.

We also wanted to further analyse the dataset to focus on **item category**.


In [ ]:

from lifelines import KaplanMeierFitter

# Analysis on cleaned target
plt.figure(figsize=(10, 6))
kmf = KaplanMeierFitter()

# Filter only targets that we want to analyze
plot_df = ttp_df[ttp_df['Target'].isin(['Women', 'Men', 'Children', 'Home and lifestyle'])]

for label, sub in plot_df.groupby('Target'):
    kmf.fit(sub["duration_days"],
            event_observed=sub["event_observed"],
            label=label)
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Clean Target")
plt.xlabel("Days since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.legend(title="Target Groups")
plt.show()

In [ ]:
# Generate survival graphs for specific columns
def plot_survival_top_n(df, column_name, top_n=10):
    plt.figure(figsize=(12, 7))
    kmf = KaplanMeierFitter()

    # Take only most frequent cathegories
    # Exclude 'n/a' and 'unclassified'
    filtered_df = df[~df[column_name].isin(['n/a', 'unclassified'])]
    top_categories = filtered_df[column_name].value_counts().head(top_n).index

    for label in top_categories:
        sub = filtered_df[filtered_df[column_name] == label]

        kmf.fit(sub["duration_days"],
                event_observed=sub["event_observed"],
                label=label)
        kmf.plot_survival_function(ci_show=True)

    plt.title(f"Survival Curve by Top {top_n} {column_name}")
    plt.xlabel("Days since first view")
    plt.ylabel("Probability of not yet purchasing")
    plt.grid(True, alpha=0.3)
    plt.legend(title=column_name, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Execution for typology
plot_survival_top_n(ttp_df, "Typology", top_n=10)

# Execution per product
plot_survival_top_n(ttp_df, "Product", top_n=10)

**Survival Curve by Clean Target (The "Who")**

This chart reveals a significant "Decision Gap" between demographics:

* **Women**: This segment displays the highest survival curve. This indicates a longer consideration phase; female users tend to browse and revisit products for a more extended period before committing to a purchase.

* **Children & Men**: These curves drop much faster in the first 48–72 hours. Purchasing behavior here is more transactional or urgent, with a much tighter window for conversion.

**Survival Curve by Top 10 Typology (The "What")**
Analyzing macro-categories provides clear operational indicators:

* **Underwear and Beachwear**: These are the "fastest" categories. The sharp initial drop indicates a lower barrier to entry and a more impulsive purchase cycle.

* **Clothing & Shoes**: These curves remain relatively high and flat after the 10-day mark. This suggests that if a customer doesn't purchase apparel or footwear within the first two weeks, the likelihood of a future conversion on that specific item diminishes significantly.

**Survival Curve by Top 10 Product (The "Detail")**

The product-level detail highlights the "Hero" products versus the "Dream" products:

Briefs & Swimsuits: These are high-velocity items. The probability of "not purchasing" drops below 60% rapidly, confirming they are essential, quick-decision purchases.

Sneakers & Dresses: These act as aspirational products. The curves stay above 80–90%, indicating that many users use the platform for "window shopping" or style research, taking much longer to pull the trigger on high-ticket or style-sensitive items.

**Strategic Conclusion**

By cleaning the data first, we can now tailor Marketing & Retargeting strategies:

Aggressive Retargeting: Should be deployed for Men and Underwear/Beachwear within the first 48 hours to capture the high conversion intent.

Nurturing Strategy: For Women and Dresses, a softer, brand-storytelling approach is better suited for their longer decision-making process.

The 30-Day Rule: Since most curves flatten out completely after 30 days, ad spend on users who haven't converted by then should be reallocated to "fresh" leads to optimize ROI.

**Session Analysis and Session KM Curves**

In this section, we move from a simple time-to-purchase description to a session-based hazard perspective. Instead of measuring only the elapsed calendar time between **first view and purchase,** we analyze the purchase process across successive sessions. For each user-item pair, we observe whether the purchase occurs in session 1, session 2, session 3, and so on, conditional on the item not having been purchased before. This allows us to estimate the session-level hazard rate, that is, the probability of conversion at each session of the customer journey.

The purpose of this section is to understand whether purchase likelihood is concentrated in the earliest sessions or persists over repeated visits.

In [ ]:
df_items = (
    df.loc[df["item_id"].notna()].copy()
    .sort_values(["user_pseudo_id", "item_id", "event_datetime"])
)

In [ ]:
# 1. Build one row per user-item-session
item_sessions = (
    df_items.groupby(["user_pseudo_id", "item_id", "session_id"], as_index=False)
    .agg(session_time=("event_datetime", "min"))
    .sort_values(["user_pseudo_id", "item_id", "session_time"])
)

# 2. Number sessions sequentially for each user-item pair
item_sessions["session_number_since_first_view"] = (
    item_sessions.groupby(["user_pseudo_id", "item_id"]).cumcount() + 1
)

# 3. Identify first purchase session for each user-item pair
purchase_session = (
    df_items[df_items["event_name"] == "purchase"]
    .sort_values(["user_pseudo_id", "item_id", "event_datetime"])
    .drop_duplicates(["user_pseudo_id", "item_id"])
    [["user_pseudo_id", "item_id", "session_id"]]
    .rename(columns={"session_id": "purchase_session_id"})
)

# 4. Merge session panel with purchase session
hazard_df = item_sessions.merge(
    purchase_session,
    on=["user_pseudo_id", "item_id"],
    how="left"
)

# 5. Flag whether purchase happens in the current session
hazard_df["purchase_in_this_session"] = (
    hazard_df["session_id"] == hazard_df["purchase_session_id"]
).fillna(False).astype(int)

# 6. Keep only sessions at risk:
#    all sessions for non-purchasers,
#    and only sessions up to the purchase session for purchasers
hazard_df["purchase_cumsum"] = (
    hazard_df.groupby(["user_pseudo_id", "item_id"])["purchase_in_this_session"].cumsum()
)

hazard_df = hazard_df[
    (hazard_df["purchase_session_id"].isna()) |
    (hazard_df.groupby(["user_pseudo_id", "item_id"])["purchase_in_this_session"]
        .cumsum()
        .shift(fill_value=0) == 0)
].copy()

# 7. Aggregate hazard by session number
session_hazard = (
    hazard_df.groupby("session_number_since_first_view")
    .agg(
        sessions_at_risk=("purchase_in_this_session", "count"),
        purchases=("purchase_in_this_session", "sum")
    )
    .reset_index()
)

# 8. Compute hazard rate
session_hazard["hazard_rate"] = (
    session_hazard["purchases"] / session_hazard["sessions_at_risk"]
)

session_hazard["hazard_rate_pct"] = (
    session_hazard["hazard_rate"] * 100
).round(2)

print(session_hazard.head(10))

This table shows the **session-level purchase hazard**, meaning the probability of buying an item in each session after the first view, conditional on not having purchased it earlier. The results suggest that conversion likelihood is concentrated in the **early sessions**: the hazard is 4.25% in session 1, rises to around 8–12% between sessions 2 and 5, and then declines in later sessions. At the same time, the number of observations at risk falls sharply across sessions, meaning that only a small share of user-item journeys continues over many repeated visits. Overall, this indicates that the **first few sessions are the most important for conversion.**

In [ ]:
plot_df = session_hazard[session_hazard["sessions_at_risk"] >= 30].copy()

plt.figure(figsize=(10, 5))
plt.plot(
    plot_df["session_number_since_first_view"],
    plot_df["hazard_rate_pct"],
    marker="o"
)

for _, row in plot_df.iterrows():
    plt.text(
        row["session_number_since_first_view"],
        row["hazard_rate_pct"] + 0.4,
        f'{row["hazard_rate_pct"]:.1f}%',
        ha="center",
        fontsize=9
    )

plt.title("Purchase Hazard by Session Number Since First View")
plt.xlabel("Session Number Since First View")
plt.ylabel("Hazard Rate (%)")
plt.xticks(plot_df["session_number_since_first_view"])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The "Peak Interest" Zone (Sessions 4 & 5): The hazard rate climbs steadily and peaks at 21.5% during the 5th session. This suggests that users typically need a period of consideration or "warming up" before they reach their maximum intent to buy.

The Sharp Drop-Off: After the 5th session, the probability of purchase crashes by more than half (to 10.2%). This indicates that if a user hasn't bought by the 5th visit, they are likely losing interest, comparing you against competitors, or simply "window shopping."

Late-Stage "Dead Zone": By session 8, the hazard rate falls to 7.7%. The longer a user stays in the funnel without converting, the less likely they are to ever do so.

To complement the time-based (days) survival analysis, we also estimated Kaplan–Meier curves using the number of sessions until purchase as the time variable.

The objective of this Kaplan-Meier model is to shift the survival analysis from chronological time (days) to user engagement intensity (the number of distinct sessions where the user interacted with the specific item).

Rather than asking 'how long' it takes for a user to purchase, we are investigating: 'how many times must a user return to view this product before they either convert or permanently lose interest?


In [ ]:
# Kaplan-Meier survival curve for Sessions to Purchase
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

plt.figure(figsize=(9, 5))

kmf.fit(
    durations=ttp_df["n_sessions_before_end"],
    event_observed=ttp_df["event_observed"],
    label="All user-item pairs"
)

kmf.plot_survival_function(ci_show=True)

plt.title("Kaplan-Meier Survival Curve: Sessions to Purchase")
plt.xlabel("Sessions since first item view")
plt.ylabel("Probability item is not yet purchased")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

# Add to cart in first session
kmf = KaplanMeierFitter()

plt.figure(figsize=(8, 5))
for label, sub in ttp_df.dropna(subset=["n_sessions_before_end", "add_to_cart_same_session"]).groupby("add_to_cart_same_session"):
    kmf.fit(
        sub["n_sessions_before_end"],
        event_observed=sub["event_observed"],
        label=f"add_to_cart={label}"
    )
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Add-to-Cart in First Session")
plt.xlabel("Sessions since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()


# Price tier
kmf = KaplanMeierFitter()

plt.figure(figsize=(8, 5))
for label, sub in ttp_df.dropna(subset=["n_sessions_before_end", "price_tier"]).groupby("price_tier"):
    kmf.fit(
        sub["n_sessions_before_end"],
        event_observed=sub["event_observed"],
        label=f"price_tier={label}"
    )
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Price Tier")
plt.xlabel("Sessions since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()


#Logged in at first view
kmf = KaplanMeierFitter()

plt.figure(figsize=(8, 5))
for label, sub in ttp_df.dropna(subset=["n_sessions_before_end", "logged_in_first_view"]).groupby("logged_in_first_view"):
    kmf.fit(
        sub["n_sessions_before_end"],
        event_observed=sub["event_observed"],
        label=f"logged_in={label}"
    )
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve by Login Status at First View")
plt.xlabel("Sessions since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()




In [ ]:
# Kaplan-Meier survival curve for Sessions to Purchase by Paid Entry
kmf = KaplanMeierFitter()

# Ensure mapping and consistency with the session-based metric name
if 'paid_entry' not in ttp_df.columns and 'user_paid_df' in globals():
    ttp_df = ttp_df.merge(user_paid_df[['user_pseudo_id', 'paid_entry']], on='user_pseudo_id', how='left')

# Use 'n_sessions_before_end' as the duration variable for consistency
plt.figure(figsize=(8, 5))
plot_km_sessions = ttp_df.dropna(subset=["n_sessions_before_end", "paid_entry"])

for label, sub in plot_km_sessions.groupby("paid_entry"):
    kmf.fit(
        sub["n_sessions_before_end"],
        event_observed=sub["event_observed"],
        label=f"paid_entry={label}"
    )
    kmf.plot_survival_function(ci_show=True)

plt.title("Survival Curve (Sessions) by Paid vs Organic Entry")
plt.xlabel("Sessions since first view")
plt.ylabel("Probability of not yet purchasing")
plt.grid(True, alpha=0.3)
plt.show()

* **Result:** The probability of "survival" (not purchasing) drops sharply within the first **5 sessions**. After approximately **10–12 sessions**, the curve flattens significantly, suggesting that if a user hasn't purchased by then, the likelihood of a future conversion from that specific product view becomes marginal.
* **Implication:** Focus marketing resources on the "Golden Window" of the first 5 interaction sessions. Beyond 10 sessions, standard retargeting yields diminishing returns.


**"Add-to-Cart" in first session**
* **Result:** There is a massive, statistically significant gap between users who add to cart in their first session and those who don't. Users with an initial `add_to_cart=1` have a survival probability that crashes to near zero almost immediately.
* **Implications:** **Intent is established early.** High-intent users don't need "convincing" over time; they need a frictionless checkout.
    * **Action:** For the `add_to_cart=0` group, the goal is "Nurturing" (education, reviews). For the `add_to_cart=1` group, the goal is "Removal of Friction" (abandoned cart emails, one-click payment options).


**Price Tier Sensitivity**
* **Result:** The `high` price tier curve (green) stays consistently above the `low` (blue) and `medium` (orange) tiers. It takes more sessions for the "survival" probability of high-priced items to drop.
* **Business Implication:** **Higher prices require higher "Consideration Effort."**
    * **Action:** Adjust your attribution windows and retargeting cadence based on price. Low-priced items should have a short, aggressive ad burst. High-priced items need a longer "drip" campaign that provides value and builds trust over more sessions (8–10) before expecting a conversion.


**Login Status at First View**
* **Result:** Logged-in users (`logged_in=1`) convert faster than guest users (`logged_in=0`), as shown by the steeper initial drop in the orange curve. However, the confidence intervals (shaded areas) eventually overlap, meaning the long-term conversion rate for those who persist is similar.
* **Business Implication:** **Recognition accelerates conversion.** Logged-in users likely have saved shipping/payment info, reducing the "pain" of the transaction.
    * **Action:** Incentivize "Login/Account Creation" early in the journey (e.g., "Login for free shipping"). This doesn't just collect data; it mathematically accelerates the velocity of the purchase journey.


**Summary for Stakeholders**
> "Our analysis confirms that the **first 5 product-specific sessions** are critical. We see that **initial intent (Add-to-Cart)** and **price** are the primary drivers of journey velocity. To maximize ROI, we should shift budget away from 'long-tail' retargeting (users with 12+ views) and reinvest it into accelerating the journey for guest users and high-price consideration segments during their first 48 hours of interaction."

### **5b. Cox Proportional Hazards Model**

In this section, we move from descriptive analysis to **formal modeling**. By estimating a survival or hazard model, we assess which **features significantly affect the speed of conversion** while accounting for censoring and heterogeneity across user-item journeys.

To rigorously evaluate the drivers of "Time to Purchase", we adopted an incremental modeling strategy using four nested Cox Proportional Hazards models.
-  **Model 1** establishes a baseline by evaluating only immediate technical and user context (e.g., login status, device).
- **Model 2** introduces product-specific factors (e.g., price tier) to account for natural variations in decision time based on the item's inherent characteristics.
- **Model 3** incorporates behavioral journey metrics and marketing acquisition data (e.g., returning visitor, paid first touch).
- **Model 4** integrates behavioral personas (user clusters) to provide a comprehensive understanding of user-specific conversion dynamics.

This stepped approach ensures mathematical fairness and allows us to observe not just which variables predict conversion, but how underlying user intent mediates demographic and product-level signals.

In [ ]:
!pip install lifelines

In [ ]:
from lifelines import CoxPHFitter

# Define all columns needed across all 4 models
all_cols = [
    "duration_days", "event_observed",
    "logged_in_first_view", "view_device",
    "price_tier",
    "returning_visitor", "paid_first_touch",
    "cluster_name"
]

# Create a working copy
cox_df = ttp_df[all_cols].copy()

# Replace inf values
cox_df = cox_df.replace([np.inf, -np.inf], np.nan)

# Handle Binary variables
bin_cols = [
    "logged_in_first_view",
    "event_observed", "returning_visitor", "paid_first_touch"
]
for col in bin_cols:
    cox_df[col] = pd.to_numeric(cox_df[col], errors="coerce").fillna(0).astype(int)

# Handle Categorical variables
cat_cols = ["view_device", "price_tier", "cluster_name"]
for col in cat_cols:
    cox_df[col] = cox_df[col].astype("object").fillna("Unknown")

# Clean Duration (must be > 0)
cox_df["duration_days"] = pd.to_numeric(cox_df["duration_days"], errors="coerce")
cox_df = cox_df[cox_df["duration_days"].notna()].copy()
cox_df["duration_days"] = cox_df["duration_days"].clip(lower=1e-6)

# Drop any remaining unclassified Targets to keep the model clean
# Removed as per user request to remove 'Target' from analysis

# Create Dummies for Categoricals (drop_first=True avoids multicollinearity)
cox_df = pd.get_dummies(cox_df, columns=cat_cols, drop_first=True)

# Convert any resulting boolean columns from get_dummies to int
for col in cox_df.columns:
    if cox_df[col].dtype == "bool":
        cox_df[col] = cox_df[col].astype(int)

# Final drop of any residual NaNs across the dataset
cox_df = cox_df.dropna().copy()

print(f"Dataset ready. Shape used for all models: {cox_df.shape}")
print(f"Purchases (events=1): {cox_df['event_observed'].sum()}")


#feature grouping (incremental approach)
# M1: Baseline (User Context & Tech)
m1_features = ["logged_in_first_view"] + [c for c in cox_df.columns if "view_device_" in c]

# M2: Product & Pricing (Baseline + Offer)
m2_features = m1_features + [c for c in cox_df.columns if "price_tier_" in c]

# M3: Journey & Marketing (Offer + Intent & Acquisition)
m3_features = m2_features + ["returning_visitor", "paid_first_touch"]

# M4: Behavioral Personas (Full Journey + Clusters)
m4_features = m3_features + [c for c in cox_df.columns if "cluster_name_" in c]


#fit the 4 models
print("\n--- Fitting Model 1: Baseline ---")
cph1 = CoxPHFitter(penalizer=0.1)
cph1.fit(cox_df[m1_features + ["duration_days", "event_observed"]],
         duration_col="duration_days", event_col="event_observed")

print("\n--- Fitting Model 2: + Product & Pricing ---")
cph2 = CoxPHFitter(penalizer=0.1)
cph2.fit(cox_df[m2_features + ["duration_days", "event_observed"]],
         duration_col="duration_days", event_col="event_observed")

print("\n--- Fitting Model 3: + Journey & Marketing ---")
cph3 = CoxPHFitter(penalizer=0.1)
cph3.fit(cox_df[m3_features + ["duration_days", "event_observed"]],
         duration_col="duration_days", event_col="event_observed")

print("\n--- Fitting Model 4: + Behavioral Clusters (Full) ---")
cph4 = CoxPHFitter(penalizer=0.1)
cph4.fit(cox_df[m4_features + ["duration_days", "event_observed"]],
         duration_col="duration_days", event_col="event_observed")


#results comparison
print("\n" + "="*60)
print("COX MODELS COMPARISON SUMMARY")
print("="*60)

# Create a comparison dataframe
comparison_results = []
models = {
    "M1 (Baseline)": cph1,
    "M2 (+ Product)": cph2,
    "M3 (+ Journey)": cph3,
    "M4 (+ Clusters)": cph4
}

for name, model in models.items():
    comparison_results.append({
        "Model": name,
        "Concordance Index ": round(model.concordance_index_, 4),
        "AIC ": round(model.AIC_partial_, 2),
        "Number of Features": len(model.params_)
    })

comp_df = pd.DataFrame(comparison_results).set_index("Model")
display(comp_df)

# Print the full statistical summary of the best model (M4)
print("\n--- FULL STATS FOR BEST MODEL (M4) ---")
cph4.print_summary()

**Cox Proportional Hazards Model: Final Results & Conversion Drivers**

Model 4 emerges as the most complete and robust specification in our analysis. By progressively enriching the model—from basic user context (M1), to product characteristics (M2), to journey variables (M3), and finally to behavioral personas (M4)—we observe a consistent improvement in performance. The final model reaches a Concordance Index of **0.75** (up from 0.60 in the baseline) and the lowest AIC, confirming that incorporating behavioral heterogeneity is essential to accurately explain time to purchase.

The most striking finding comes from the inclusion of behavioral clusters. These variables are among the strongest predictors in the model and clearly differentiate how quickly users convert. Compared to the baseline group (Cluster A), both Cluster B and Cluster C exhibit significantly higher hazard rates. In particular, users in **Cluster C convert almost 2.5 times faster (HR = 2.48)**, confirming their highly impulsive nature: while they are generally low-engagement users, when they do decide to purchase, they do so very quickly. Similarly, **Cluster B users convert about 60% faster (HR = 1.61)**, reflecting a more balanced behavior—some level of exploration, but still relatively quick decision-making once interest is established.

Importantly, while Cluster B users may appear slower in descriptive analyses due to longer average paths to purchase, the Cox model reveals that, conditional on comparable behavioral and contextual factors, they actually convert more quickly once they reach a similar decision stage.

By contrast, Cluster A remains the slowest segment, reinforcing the idea that the most engaged users are also the most deliberate, likely due to higher involvement and more careful evaluation before purchasing.

Product characteristics also play a clear role in shaping the speed of conversion. Lower-priced items are associated with significantly faster purchases (**HR = 1.59**), while medium-priced items also show a positive but more moderate effect (**HR = 1.23**). This confirms that **price acts as a key source of friction**: as the financial commitment increases, users require more time to evaluate their decision.

User context further reinforces the importance of reducing friction in the early stages of the journey. Users who are **logged in at their first interaction convert substantially faster (HR = 1.42)**, suggesting that familiarity with the brand and reduced checkout barriers accelerate the process. Similarly, **mobile users convert slightly faster (HR = 1.15)**, consistent with more immediate, convenience-driven behavior.

At the same time, the model reveals an important and somewhat counterintuitive pattern regarding loyalty. Being a **returning visitor significantly reduces the hazard rate (HR = 0.62)**, meaning that these users take longer to convert from the moment they first view a product.

Finally, acquisition channel does not appear to significantly influence the speed of conversion. The effect of **paid first touch is small and not statistically meaningful (HR ≈ 0.95)**.

Taken together, these results point to a fundamental tension between **speed and depth of engagement**. Faster conversions are typically driven by low-friction conditions—lower prices, logged-in users, and more impulsive behavioral profiles—whereas slower conversions are associated with more engaged, returning users who follow a longer and more deliberate path to purchase. This implies that optimizing conversion cannot rely on a single strategy: instead, Versace should recognize and accommodate these different behavioral patterns, accelerating quick decisions where possible while supporting longer, high-consideration journeys for its most engaged users.

## **6. Delayed Co-purchasing Behavior**

### **6a. Viewed pairs**
This section studies whether combinations of items viewed together in one session are later purchased together in a different session. We begin by identifying product pairs that appeared together during browsing behavior.

We start the analysis building two datasets:


*   View Pair: each row represents a specific user viewing a specific pair of items together in a specific session
* First Pair Exposure: each row represents the first session in which each user viewed a given item pair together



In [ ]:
df.columns

In [ ]:
from itertools import combinations

df['event_datetime'] = pd.to_datetime(df['event_timestamp'].astype(int), unit='us')

view_sessions = (
    df_items[df_items["event_name"] == "view_item"]
    .groupby(["user_pseudo_id", "session_id"])["item_id"]
    .apply(lambda x: sorted(set(x.dropna())))
    .reset_index(name="viewed_items")
)

view_sessions["item_pairs"] = view_sessions["viewed_items"].apply(
    lambda items: list(combinations(items, 2)) if len(items) >= 2 else []
)

view_pairs = view_sessions.explode("item_pairs").dropna(subset=["item_pairs"]).copy()

session_start_times = (
    df.groupby(["user_pseudo_id", "session_id"], as_index=False)
    .agg(session_time=("event_datetime", "min"))
)

view_pairs = view_pairs.merge(
    session_start_times,
    on=["user_pseudo_id", "session_id"],
    how="left"
)

view_pairs[["item_1", "item_2"]] = pd.DataFrame(
    view_pairs["item_pairs"].tolist(),
    index=view_pairs.index
)

# Build item -> detailed categories and price lookup
item_lookup = (
    df_items[["item_id", "item_category", "item_price"]]
    .dropna(subset=["item_id"])
    .drop_duplicates(subset=["item_id"])
)

# Compute global price tiers based on all unique products
item_lookup["price_tier"] = pd.qcut(
    item_lookup["item_price"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

# Apply the parser to create target, typology, and product
item_lookup[["target", "typology", "product"]] = item_lookup["item_category"].apply(parse_categories_refined)

# Merge attributes for item_1
view_pairs = view_pairs.merge(
    item_lookup.rename(columns={
        "item_id": "item_1",
        "target": "target_1",
        "typology": "typology_1",
        "product": "product_1",
        "price_tier": "price_tier_1"
    }).drop(columns=["item_category", "item_price"]),
    on="item_1",
    how="left"
)

# Merge attributes for item_2
view_pairs = view_pairs.merge(
    item_lookup.rename(columns={
        "item_id": "item_2",
        "target": "target_2",
        "typology": "typology_2",
        "product": "product_2",
        "price_tier": "price_tier_2"
    }).drop(columns=["item_category", "item_price"]),
    on="item_2",
    how="left"
)

first_pair_exposure = (
    view_pairs.sort_values("session_time")
    .drop_duplicates(["user_pseudo_id", "item_pairs"])
    .rename(columns={
        "session_time": "pair_first_view_time",
        "session_id": "pair_first_view_session"
    })
)

first_pair_exposure = first_pair_exposure[
    [
        "user_pseudo_id",
        "item_pairs",
        "item_1",
        "item_2",
        "target_1",
        "target_2",
        "typology_1",
        "typology_2",
        "product_1",
        "product_2",
        "price_tier_1",
        "price_tier_2",
        "pair_first_view_session",
        "pair_first_view_time"
    ]
].copy()

first_pair_exposure.head()

In [ ]:
view_pairs.head()

In [ ]:
print("Number of sessions with at least one viewed item:", len(view_sessions))
print("Number of session-item pairs generated:", len(view_pairs))
print("Number of unique first pair exposures:", len(first_pair_exposure))

print(first_pair_exposure.head())

In [ ]:
import pandas as pd

def get_top_pairs(df, col1, col2, title):
    print(f"\n--- Most frequent {title} pairs viewed ---")
    # Create sorted tuples to treat (A, B) and (B, A) as identical
    pairs = df.apply(
        lambda row: tuple(sorted([str(row[col1]), str(row[col2])])),
        axis=1
    )
    counts = pairs.value_counts().head(10)
    print(counts)

    # Calculate percentage of same-category pairs for this dimension
    same_pct = (df[col1] == df[col2]).mean() * 100
    print(f"Percentage of times {title} 1 and 2 are the same: {same_pct:.2f}%")

# Analyze Target pairs
get_top_pairs(view_pairs, 'target_1', 'target_2', 'Target')

# Analyze Typology pairs
get_top_pairs(view_pairs, 'typology_1', 'typology_2', 'Typology')

# Analyze Product pairs
get_top_pairs(view_pairs, 'product_1', 'product_2', 'Product')

**Hierarchical Co-viewing Analysis Results**

The transition to a granular three-tier hierarchy (**Target**, **Typology**, **Product**) reveals distinct browsing behaviors across the customer journey:

*   **Target Level (76.75% consistency):** Browsing is highly demographic-centric. Users stay within the same segment (e.g., Men-Men) nearly 77% of the time, though the most frequent cross-segment behavior occurs between **Men** and **Women**.
*   **Typology Level (53.66% consistency):** There is a balanced mix of comparison and exploration. Over half of the sessions focus on the same macro-category (e.g., Clothing-Clothing), but significant cross-exploration exists between categories like **Clothing** and **Shoes**.
*   **Product Level (26.37% consistency):** At the most granular level, exploration dominates. Only about 26% of co-viewed pairs consist of the same product type, suggesting that users are often looking to build outfits or discover diverse items (e.g., **T-shirts** and **Sneakers**) rather than just comparing identical types.

These findings suggest that while brand loyalty and demographic focus are strong, the Versace audience utilizes the platform for broader discovery and lifestyle curation across different product typologies.

### **6b. Delayed co purchasing-purchased pairs**
Here we identify pairs of products that were **actually purchased** together in the same transaction. This allows us to compare early browsing bundles with final basket composition.

In [ ]:
# Build one row per purchase transaction with purchased items
purchase_tx = (
    df_items[df_items["event_name"] == "purchase"]
    .groupby(["user_pseudo_id", "transaction_id", "session_id"])["item_id"]
    .apply(lambda x: sorted(set(x.dropna())))
    .reset_index(name="purchased_items")
)

purchase_tx = purchase_tx[purchase_tx["transaction_id"].notna()].copy()

# Generate item pairs within each purchase transaction
purchase_tx["item_pairs"] = purchase_tx["purchased_items"].apply(
    lambda items: list(combinations(items, 2)) if len(items) >= 2 else []
)

purchase_pairs = purchase_tx.explode("item_pairs").dropna(subset=["item_pairs"]).copy()

# Split the pair into two item columns
purchase_pairs[["item_1", "item_2"]] = pd.DataFrame(
    purchase_pairs["item_pairs"].tolist(),
    index=purchase_pairs.index
)

# Add hierarchical category and price information for both items
# Re-using the lookup logic with refined parsing and price tiers
item_lookup_full = (
    df_items[["item_id", "item_category", "item_price"]]
    .dropna(subset=["item_id"])
    .drop_duplicates(subset=["item_id"])
)

# Compute global price tiers
item_lookup_full["price_tier"] = pd.qcut(
    item_lookup_full["item_price"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

# Parse into Target, Typology, Product
item_lookup_full[["target", "typology", "product"]] = item_lookup_full["item_category"].apply(parse_categories_refined)

# Merge attributes for item_1
purchase_pairs = purchase_pairs.merge(
    item_lookup_full.rename(columns={
        "item_id": "item_1",
        "target": "target_1",
        "typology": "typology_1",
        "product": "product_1",
        "price_tier": "price_tier_1"
    }).drop(columns=["item_category", "item_price"]),
    on="item_1",
    how="left"
)

# Merge attributes for item_2
purchase_pairs = purchase_pairs.merge(
    item_lookup_full.rename(columns={
        "item_id": "item_2",
        "target": "target_2",
        "typology": "typology_2",
        "product": "product_2",
        "price_tier": "price_tier_2"
    }).drop(columns=["item_category", "item_price"]),
    on="item_2",
    how="left"
)

#Add purchase session time
purchase_session_times = (
    df.groupby(["user_pseudo_id", "session_id"], as_index=False)
      .agg(session_time=("event_datetime", "min"))
)

purchase_pairs = purchase_pairs.merge(
    purchase_session_times,
    on=["user_pseudo_id", "session_id"],
    how="left"
)

purchase_pairs = purchase_pairs.rename(columns={"session_time": "pair_purchase_time"})

purchase_pairs.head()

In [ ]:
print("Number of transactions with at least one purchased item:", len(purchase_tx))
print("Number of purchased item-pairs generated:", len(purchase_pairs))
print("Transactions with at least 2 purchased items:", (purchase_tx["purchased_items"].apply(len) >= 2).sum())

print(purchase_pairs[["user_pseudo_id", "transaction_id", "session_id", "item_pairs", "pair_purchase_time"]].head())

In [ ]:
# Analyze consistency within purchased pairs across the hierarchy
def get_purchase_pair_stats(df, col1, col2, title):
    print(f"\n--- Most frequent {title} pairs PURCHASED ---")
    # Treat (A, B) and (B, A) as identical
    pairs = df.apply(
        lambda row: tuple(sorted([str(row[col1]), str(row[col2])])),
        axis=1
    )
    counts = pairs.value_counts().head(10)
    print(counts)

    # Calculate percentage of same-category pairs
    same_pct = (df[col1] == df[col2]).mean() * 100
    print(f"Consistency Rate ({title}): {same_pct:.2f}%")

# Analyze Target consistency in purchases
get_purchase_pair_stats(purchase_pairs, 'target_1', 'target_2', 'Target')

# Analyze Typology consistency in purchases
get_purchase_pair_stats(purchase_pairs, 'typology_1', 'typology_2', 'Typology')

# Analyze Product consistency in purchases
get_purchase_pair_stats(purchase_pairs, 'product_1', 'product_2', 'Product')

 **Hierarchical Co-purchase Consistency Results**

The transition from browsing to purchasing reveals a **'Conversion Funnel of Consistency'**, where final baskets are significantly more focused than initial exploration:

*   **Target Level (94.08% consistency):** While users may browse across demographics, they almost exclusively buy for a single target group (e.g., Men or Children) once they reach the checkout.
*   **Typology Level (70.72% consistency):** Consistency increases significantly compared to browsing (53%), driven heavily by multi-item purchases in categories like *Underwear*, *Beachwear*, and *Baby* clothing.
*   **Product Level (37.28% consistency):** Even at the most granular level, users are more likely to buy identical product types together (e.g., multiple *Decor* items or *Briefs*) than they are to view them together.

**Strategic Insight:** Browsing is for discovery and 'outfit building' across categories, but purchasing is primarily for replenishment or depth within a single demographic and category.

### **6c. Delayed co-purchase probability**
In this section, we measure the probability that an exact pair of items first viewed together is later bought together in a subsequent session. This helps evaluate whether early cross-item consideration translates into delayed joint purchase behavior.

In [ ]:
pair_analysis = first_pair_exposure.merge(
    purchase_pairs[["user_pseudo_id", "item_pairs", "pair_purchase_time", "session_id"]],
    on=["user_pseudo_id", "item_pairs"],
    how="left",
    suffixes=("", "_purchase")
)

#variables to capture if the item couple is purchased or not
pair_analysis["delayed_copurchase"] = (
    pair_analysis["pair_purchase_time"].notna() &
    (pair_analysis["pair_purchase_time"] > pair_analysis["pair_first_view_time"])
).astype(int)

In [ ]:
delayed_rate = pair_analysis["delayed_copurchase"].mean() * 100
print(f"Delayed exact pair co-purchase rate: {delayed_rate:.2f}%")

In [ ]:
pair_summary = pd.Series({
    "first_pair_exposures": len(pair_analysis),
    "delayed_exact_copurchases": pair_analysis["delayed_copurchase"].sum(),
    "delayed_copurchase_rate_pct": round(pair_analysis["delayed_copurchase"].mean() * 100, 2)
})

print(pair_summary)

In this step, we compare the first observed exposure to each viewed item-pair with the set of item-pairs later purchased together in order to measure delayed exact co-purchasing.
A pair is classified as a delayed co-purchase when the same user buys the exact same pair of items together, after having previously viewed them together.

The results show that, out of 17,341 first pair exposures, **only 166 lead to a delayed exact co-purchase, corresponding to a rate of just 0.97%**. This suggests that although users often explore multiple products together during browsing, these exact combinations are only rarely converted into joint purchases at a later point in time. It is important to interpret these results carefully, as they are based on a relatively low number of observations for an e-commerce platform.


We now test whether delayed co-purchasing is more likely when the two viewed items belong to the same product category (divided in target, typology and product). It helps us understand whether **category similarity** increases the stability of product pair consideration over time. To do this, each viewed pair is classified according to whether both items share the same category, and the delayed co-purchase rate is then compared across the two groups.

In [ ]:
# Create mapping dictionaries for each hierarchy level using existing lookup table
item_target_map = item_lookup_full.set_index('item_id')['target'].to_dict()
item_typology_map = item_lookup_full.set_index('item_id')['typology'].to_dict()
item_product_map = item_lookup_full.set_index('item_id')['product'].to_dict()

In [ ]:
def check_same_attr(pair, attr_map):
    i1, i2 = pair
    a1, a2 = attr_map.get(i1), attr_map.get(i2)
    return int(a1 == a2) if (a1 is not None and a2 is not None) else np.nan

# Calculate consistency flags for each level
pair_analysis['same_target'] = pair_analysis['item_pairs'].apply(lambda x: check_same_attr(x, item_target_map))
pair_analysis['same_typology'] = pair_analysis['item_pairs'].apply(lambda x: check_same_attr(x, item_typology_map))
pair_analysis['same_product'] = pair_analysis['item_pairs'].apply(lambda x: check_same_attr(x, item_product_map))

In [ ]:
for level in ['same_target', 'same_typology', 'same_product']:
    print(f"\n--- Delayed Co-purchase Probability: {level.replace('_', ' ').title()} ---")
    summary = pair_analysis.groupby(level)['delayed_copurchase'].agg(['count', 'sum', 'mean']).reset_index()
    summary['mean'] = (summary['mean'] * 100).round(2)
    print(summary)

Main results:

* **Same Target** (1.18% vs 0.32%): Items viewed within the same target demographic are significantly more likely to be purchased together later. This suggests that while browsing might be broad, the 'delayed' intent is usually demographic-specific.

* **Same Typology** (1.08% vs 0.85%): There is a slight increase in probability when items share the same category group, indicating that users return to buy from categories they were previously comparing.

* **Same Product** (0.88% vs 1.00%): At the most granular level, users are slightly less likely to buy two identical product types together later than different products. This reinforces the idea that delayed co-purchasing is often driven by 'outfit-building' or diverse discovery rather than stockpiling the same item type.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Ensure price_tier attributes are in pair_analysis
price_map = item_lookup.set_index('item_id')['price_tier'].to_dict()

pair_analysis['price_tier_1'] = pair_analysis['item_1'].map(price_map)
pair_analysis['price_tier_2'] = pair_analysis['item_2'].map(price_map)

# Create a helper to get consistent price tier pairs
def get_tier_pair(row):
    t1 = str(row['price_tier_1'])
    t2 = str(row['price_tier_2'])
    if t1 == 'nan' or t2 == 'nan':
        return np.nan
    return tuple(sorted([t1, t2]))

pair_analysis['price_tier_pair'] = pair_analysis.apply(get_tier_pair, axis=1)

# Calculate the probability of co-purchase per price tier combination
# Filter out pairs with missing price tiers
valid_pairs = pair_analysis.dropna(subset=['price_tier_pair'])

price_tier_matrix_data = valid_pairs.groupby('price_tier_pair').agg(
    exposures=('delayed_copurchase', 'count'),
    purchases=('delayed_copurchase', 'sum'),
    probability=('delayed_copurchase', 'mean')
).reset_index()

# Pivot the data for a heatmap visualization
price_tier_matrix_data[['Tier_A', 'Tier_B']] = pd.DataFrame(price_tier_matrix_data['price_tier_pair'].tolist(), index=price_tier_matrix_data.index)

# Create a symmetric matrix for the heatmap
matrix_pivot = price_tier_matrix_data.pivot(index='Tier_A', columns='Tier_B', values='probability')

# Ensure symmetry
for i in matrix_pivot.index:
    for j in matrix_pivot.columns:
        if pd.isna(matrix_pivot.loc[i, j]) and j in matrix_pivot.index and i in matrix_pivot.columns:
            matrix_pivot.loc[i, j] = matrix_pivot.loc[j, i]

# 5. Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(matrix_pivot * 100, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={'label': 'Co-purchase Probability (%)'})
plt.title("Price Tier Co-purchase Probability Matrix (Viewed -> Purchased)")
plt.xlabel("Item 2 Price Tier")
plt.ylabel("Item 1 Price Tier")
plt.show()

print("--- Summary Table ---")
display(price_tier_matrix_data.sort_values('probability', ascending=False))

The heatmap and summary table reveal some interesting trends in consumer behavior:

* **High Affinity in Lower Tiers**: The most successful co-purchase combination is (Low, Low) with a probability of 2.72%. This suggests that users looking at entry-level items are more likely to build baskets with multiple items from the same price bracket.
* **Mixed Intent**: Combinations like (Low, Medium) also show a relatively healthy conversion rate (1.24%), indicating some level of cross-tier exploration.
* **High-Tier Stability**: Conversely, (High, High) pairings have the lowest co-purchase probability (0.09%). This aligns with our previous findings that higher-priced items require much deeper consideration and are less likely to be bought in spontaneous bundles.

This matrix provides a clear roadmap for cross-selling strategies—focusing on volume in the low-tier bundles while recognizing that high-tier purchases remain more isolated events.

In [ ]:
# 1. Calculate the average revenue for each price tier combination
# We take the sum of prices from the original lookup for both items in the pair

item_price_map = item_lookup_full.set_index('item_id')['item_price'].to_dict()

valid_pairs['pair_revenue'] = valid_pairs['item_1'].map(item_price_map) + valid_pairs['item_2'].map(item_price_map)

# 2. Aggregate revenue data by price tier combination
revenue_impact = valid_pairs.groupby('price_tier_pair').agg(
    avg_pair_value=('pair_revenue', 'mean'),
    total_exposures=('delayed_copurchase', 'count'),
    total_purchases=('delayed_copurchase', 'sum')
).reset_index()

# 3. Calculate Potential Revenue (Expected Value per Exposure)
# Expected Value = Probability * Average Value
revenue_impact['conversion_rate'] = revenue_impact['total_purchases'] / revenue_impact['total_exposures']
revenue_impact['expected_value_per_exposure'] = revenue_impact['conversion_rate'] * revenue_impact['avg_pair_value']
revenue_impact['total_generated_revenue'] = revenue_impact['total_purchases'] * revenue_impact['avg_pair_value']

# 4. Display the enriched analysis
print("--- Revenue Impact of Price Tier Combinations ---")
display(revenue_impact[['price_tier_pair', 'avg_pair_value', 'conversion_rate', 'expected_value_per_exposure', 'total_generated_revenue']].sort_values('total_generated_revenue', ascending=False))

# Determine the top contributor for the textual analysis
top_revenue_pair = revenue_impact.sort_values('total_generated_revenue', ascending=False).iloc[0]['price_tier_pair']


**Revenue Impact Analysis: Volume vs. Expected Value**

While (Low, Low) combinations drive volume and boast the highest conversion rate (2.72%), an analysis of total revenue contribution reveals a much more nuanced reality regarding profitability.

* **The "Medium-Medium" Sweet Spot:** Despite having less than half the conversion rate of low-tier items (1.12%), the **(Medium, Medium)** pairing actually holds the highest **Expected Value per Exposure (€7.93)**. This indicates that driving traffic to co-viewed medium-tier items is statistically the most profitable retargeting strategy on a per-session basis, generating a highly competitive €20.4k in total delayed revenue.
* **Volume Dominance:** The **(Low, Low)** tier remains the highest overall revenue generator (€22.7k). Its lower average basket value (€252) is more than compensated by its sheer conversion velocity and volume.
* **The High-Tier Friction:** Pairings involving High-tier items confirm that extreme luxury is rarely an impulsive bundled purchase. Even with a massive average pair value of €2,340, the **(High, High)** combination suffers from a near-zero conversion rate (0.09%), leading to poor total generated revenue (€7k) and a low expected value per exposure (€2.11).


### **6d. Time to Delayed Co-Purchase**

We are now transitioning from analyzing if a delayed co-purchase happens to analyzing when it happens. By merging your initial pair exposure data with the actual purchase timestamps, we have created a timeline for every successful delayed conversion.

This code will isolate the 166 successful "delayed" transactions, calculate the exact number of days it took the user to make the purchase after their first view, and map these delays against product price tiers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


#calculating the delay
# Filter the dataset to keep ONLY the successful delayed co-purchases
delayed_pairs = pair_analysis[pair_analysis['pair_purchase_time'].notna()].copy()

# Ensure the timestamp columns are in datetime format
delayed_pairs['pair_first_view_time'] = pd.to_datetime(delayed_pairs['pair_first_view_time'])
delayed_pairs['pair_purchase_time'] = pd.to_datetime(delayed_pairs['pair_purchase_time'])

# Calculate the delay in exact days
delayed_pairs['delay_days'] = (delayed_pairs['pair_purchase_time'] - delayed_pairs['pair_first_view_time']).dt.total_seconds() / (24 * 3600)

# Filter out immediate purchases (> 1 hour)
delayed_pairs = delayed_pairs[delayed_pairs['delay_days'] > 0.04].copy()

global_median_delay = delayed_pairs['delay_days'].median()
print(f"Global Median Delay for Co-Purchasing: {global_median_delay:.2f} days")


#formatting price tier combinations
def combine_tiers(row):
    tiers = sorted([str(row['price_tier_1']), str(row['price_tier_2'])])
    return f"{tiers[0]} - {tiers[1]}"

delayed_pairs['tier_combination'] = delayed_pairs.apply(combine_tiers, axis=1)

#VISUALIZATION 1: GLOBAL DISTRIBUTION
plt.figure(figsize=(10, 6))
sns.histplot(delayed_pairs['delay_days'], bins=30, kde=True, color='#2c3e50')
plt.axvline(global_median_delay, color='#e74c3c', linestyle='dashed', linewidth=2,
            label=f'Median: {global_median_delay:.1f} days')
plt.title('Time to Delayed Co-Purchase Distribution', fontsize=15, pad=15)
plt.xlabel('Days between First Co-View and Co-Purchase', fontsize=12)
plt.ylabel('Number of Occurrences', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

#VISUALIZATION 2: DELAY BY PRICE TIER (MODIFIED FOR VISIBILITY)
plt.figure(figsize=(12, 6))
order = delayed_pairs.groupby('tier_combination')['delay_days'].median().sort_values().index

# Highlighting the median line with medianprops
sns.boxplot(
    x='tier_combination',
    y='delay_days',
    data=delayed_pairs,
    order=order,
    palette='viridis',
    boxprops=dict(alpha=0.6),
    medianprops={'color': 'orange', 'linewidth': 3}  # Bright, thick median line
)

# Overlay a swarmplot
sns.swarmplot(
    x='tier_combination',
    y='delay_days',
    data=delayed_pairs,
    order=order,
    color='black',
    alpha=0.4,
    size=4
)

plt.title('Time to Delayed Co-Purchase by Price Tier Combination (Orange Line = Median)', fontsize=15, pad=15)
plt.xlabel('Price Tier Combination', fontsize=12)
plt.ylabel('Delay (Days - Log Scale)', fontsize=12)
plt.yscale('log')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

**The Dynamics of Delayed Conversion: Timing and Price Influence**

* **The Global Consideration Window (The 48-Hour Rule):** The overall median time between the first co-view and the final co-purchase is exceptionally short (less than 24 hours). The distribution is extremely right-skewed, meaning the vast majority of delayed bundles are closed within the first 1 to 2 days. After the 48-hour mark, the probability of a delayed co-purchase drops dramatically.
* **Impulse vs. Consideration by Price Tier:** Analyzing the delay by price combination reveals distinct decision-making speeds. Bundles consisting of **(Medium - Medium)** and **(Low - Low)** items convert the fastest, often within hours or by the very next day. Conversely, combinations involving higher tiers, such as **(High - Medium)**, exhibit a visibly longer consideration phase, with the interquartile range stretching from 1 to roughly 2.5 days.
* **The "High-Tier" Rarity:** Notably, there are almost zero occurrences of delayed co-purchasing for **(High - High)** and **(High - Low)** combinations. This implies that extreme luxury bundling is almost never a delayed, pondered decision; it either happens immediately in the first session or not at all.


While temporal dynamics and price tiers dictate *when* and *if* a delayed bundle is profitable, the strongest predictor of a delayed co-purchase is intermediate intent. We now analyze the portion of purchased pairs added to cart together in the same session

In [ ]:
# 1. Identify pairs that were added to the cart together in the SAME session
cart_sessions = (
    df_items[df_items["event_name"] == "add_to_cart"]
    .groupby(["user_pseudo_id", "session_id"])["item_id"]
    .apply(lambda x: sorted(set(x.dropna())))
    .reset_index(name="carted_items")
)

cart_sessions["cart_pairs"] = cart_sessions["carted_items"].apply(
    lambda items: list(combinations(items, 2)) if len(items) >= 2 else []
)

cart_pairs_df = cart_sessions.explode("cart_pairs").dropna(subset=["cart_pairs"]).copy()

# 2. Check how many of the 166 delayed co-purchases were added to cart in their first view session
# We merge pair_analysis (filtered for delayed_copurchase == 1) with cart_pairs_df
delayed_success = pair_analysis[pair_analysis['delayed_copurchase'] == 1].copy()

cart_intent_check = delayed_success.merge(
    cart_pairs_df.rename(columns={'cart_pairs': 'item_pairs', 'session_id': 'pair_first_view_session'}),
    on=['user_pseudo_id', 'item_pairs', 'pair_first_view_session'],
    how='left',
    indicator=True
)

# 3. Calculate statistics
num_delayed = len(delayed_success)
num_with_cart = (cart_intent_check['_merge'] == 'both').sum()
pct_with_cart = (num_with_cart / num_delayed) * 100

print(f"--- Analysis: The Role of the Cart (Intermediate Intent) ---")
print(f"Total Delayed Co-purchases: {num_delayed}")
print(f"Pairs added to cart together in initial session: {num_with_cart}")
print(f"Percentage: {pct_with_cart:.2f}%")


The analysis results are quite striking: 81.3% of all delayed co-purchases (135 out of 166) were actually added to the cart together during their very first session.

This validates a major strategic hypothesis: for the vast majority of 'delayed' bundles, the user didn't just browse them—they expressed a high-intent desire to buy them right from the start. This makes 'Abandoned Bundle' retargeting one of the most promising growth opportunities, as these users have already done the heavy lifting of product discovery and selection.

This validates a major strategic hypothesis: for the vast majority of 'delayed' bundles, the user didn't just browse them—they expressed a high-intent desire to buy them right from the start. This makes 'Abandoned Bundle' retargeting one of the most promising growth opportunities, as these users have already done the heavy lifting of product discovery and selection.

## **7. Key Results & Business Recommendations**


### **7a. Key Results**
 **1. Time-to-Purchase Distribution, Conversion & Visual Inspection**

The TTP dataset comprises 6,017 user–item observation episodes built at the *user_pseudo_id–item* level over a 60-day window (Italy only, June–July 2025), of which 1,144 convert (≈19% item-level conversion rate). An important caveat applies: because *user_pseudo_id* is a cookie-based, device-level identifier, users who switch devices or clear their browser are counted as distinct individuals. Login data is available for only 11.5% of sessions, making *user_pseudo_id* the most complete - though imperfect - proxy for unique visitors. TTP metrics therefore reflect the “final mile” of the purchase journey rather than total consideration time.

Among converters, the distribution is strongly right-skewed: the median TTP is approximately 22 minutes, while the mean reaches ~1.57 days. Breaking purchases into time buckets: 59.8% close within one hour, 17.9% within 24 hours, 15.8% within 7 days, and only 6.5% beyond 7 days — confirming that roughly 78% of purchases complete within one day.

Category-level patterns are pronounced. By **Target**, Men account for the largest share (46.9%) and convert fastest; Women (30.6%) show higher median TTP and more dispersed distributions, reflecting longer deliberation. By **Typology**, Underwear/Beachwear (29.3% of transactions) has the shortest TTP; Clothing and Shoes require more sessions and exhibit wider IQR. By **Price Tier**, low-price items convert both more often and faster; the consideration window widens substantially at medium and high price points. Regarding **Login Status**, logged-in users show a consistently lower and less dispersed TTP distribution compared to anonymous visitors, indicating that reducing account friction accelerates conversion. Finally, **Returning visitors**, counterintuitively, display longer TTP than new users, suggesting deeper deliberation rather than faster repeat purchases.

**2. Key Drivers of Conversion Speed**

Driver summary tables confirm a coherent pattern: variables signalling high prior intent are associated with both higher conversion rates and shorter TTP:

- **Mobile (vs. desktop):** highest conversion rate and shorter median duration across all sessions.  
- **Add-to-cart (same session):** the single largest behavioural predictor — users who cart an item in the first session convert at dramatically higher rates than pure browsers.  
- **Price tier:** inverse relationship with conversion rate and speed; higher price = lower frequency and longer TTP.  
- **Returning visitor:** lower conversion rate and longer median duration than new visitors, pointing to extended multi-session curation behaviour.

**3. Campaign / UTM Analysis**

UTM parameters extracted from page URLs reveal that paid traffic plays multiple, functionally distinct roles in the purchase journey. Overall, users with at least one paid-session exposure show higher average and median TTP than purely organic users, indicating that paid channels often support consideration rather than drive immediate conversion. However, timing within the journey is critical: when paid enters as the **first touch** (session rank = 1), journeys are shorter — consistent with an acquisition role targeting high-intent users. In contrast, when paid appears only from **session 3 onward**, it acts more as a remarketing or reactivation channel, with notably longer purchase paths.

A meaningful share of converters also complete their purchase within a paid session, confirming that campaigns contribute both to mid-funnel consideration and to final conversion. At the same time, **source/medium heterogeneity** is substantial: Google Shopping and affiliate traffic are associated with the shortest TTP (clear demand capture), while social and email channels correlate with longer journeys (inspiration and nurturing roles).

These results suggest that evaluating all paid channels under a single last-click framework leads to systematic misallocation of budget, as upper- and mid-funnel channels appear less effective than they actually are. A more accurate approach is to explicitly differentiate acquisition and nurturing roles. In addition, landing page design should be aligned with traffic intent: high-intent search and shopping traffic benefits from frictionless, conversion-oriented experiences, while social and email traffic requires richer content and product discovery elements to support longer consideration journeys.

**4. Behavioural Segmentation**

K-means clustering on three behavioural features — *average item price viewed*, *number of distinct other items viewed*, and *number of item re-views* —  yields three clearly differentiated segments:

- **Cluster A — High-Value Power Shoppers:**  
  Highest total revenue (€183k), widest catalogue exploration (avg. 6.5 distinct items), premium price targets, and the highest mean transactions per person (1.06). These users exhibit deliberate, high-consideration purchase behaviour and represent the most valuable converters.

- **Cluster B — Cautious Researchers:**  
  Mid-tier spend (avg. €253/person), strongest focus on specific products (avg. 3.31 re-views of the same item), and the longest mean TTP (2.72 days). This segment reflects intensive evaluation and comparison before purchase.

- **Cluster C — Impulse Entry-Buyers:**  
  Lowest spend (avg. €166/person), minimal catalogue exploration, but the fastest conversion speed (mean 0.43 days). These users display low-friction, quick decision-making behaviour.

**5. Kaplan–Meier Survival Analysis**

The overall survival curve drops sharply within the first 1–2 days and then flattens, consistent with a strongly front-loaded TTP distribution. Stratified survival curves highlight substantial behavioural differences:

- **Add-to-cart (first session):** produces a near-vertical drop in survival (rapid conversion), whereas non-carted items remain largely unconverted.  
- **Price tier:** clearly ordered survival curves (low < medium < high), with higher-priced items sustaining longer consideration periods.   
- **Behavioural clusters:** Cluster C drops fastest (impulse), Cluster A shows the most persistence (deliberate), confirming segmentation validity.
- **Category-level** show that entry-level products (e.g., fragrances or small accessories) exhibit faster “decay” (quicker conversion), while high-investment items (e.g., leather handbags or outerwear) maintain higher survival probabilities over longer periods, reflecting extended evaluation cycles:
    - **Faster conversion (within 48–72h):** Men, Underwear/Beachwear  
    - **Longer consideration (flattening after 10–14 days):** Women, Clothing/Shoes  

Session-based Kaplan–Meier analysis further reveals that purchase probability peaks around **sessions 4–5 (≈21.5% hazard rate)** and declines sharply after **session 10–12**, defining a practical *“Golden Window”* for retargeting interventions.

**6. Cox Proportional Hazards Model**

We compared four versions of the model, and **Model 4 (the final version)** proved to be the most accurate. This confirms that grouping users into behavioural clusters is essential for predicting *when* a purchase will happen. Key findings from Model 4 include:

- **Cluster C (vs. A): HR = 2.48** – Impulse buyers are roughly **2.5× faster** to convert than core shoppers.  
- **Cluster B (vs. A): HR = 1.61** – “Researchers” are **60% more likely** to convert at any given moment than core shoppers. Despite longer average TTP, they are more *determined* to complete the journey. In contrast, many “Power Shoppers” (Cluster A) browse premium items extensively but often do not purchase, lowering their overall hazard.  
- **Price Tiers:** Low (HR = 1.59) and Medium (HR = 1.23) – Price acts as a major **speed barrier**; lower prices significantly accelerate conversion.  
- **Logged-in at first view: HR = 1.42** – Early user recognition speeds up purchase by **42%**.
- **Mobile Device: HR = 1.15** – Mobile users convert slightly faster than desktop users.
- **Returning Visitor: HR = 0.62** – Returning users take significantly longer to convert, reflecting extended, multi-session research rather than quick repeat purchasing. This effect may also be partially driven by measurement limitations: *user_pseudo_id* is a cookie-based identifier and not fully reliable, meaning the same user across devices or sessions may be counted as different individuals, potentially inflating observed journey length.

**7. Delayed Co-Purchasing Behaviour**

Out of 17,341 first pair-view exposures, only 166 result in a delayed exact co-purchase (**0.97% conversion rate**), confirming that co-viewing rarely translates directly into co-purchasing. Browsing is primarily exploratory, while purchasing is far more concentrated:

- **Browsing patterns:** 76.75% of viewed pairs share the same Target, with substantial cross-category exploration.  
- **Purchase patterns:** 94.08% same-Target consistency and 70.72% same-Typology concentration, indicating strong narrowing at conversion.

**Category similarity effects:**
- Same-Target pairs: **1.18% conversion** vs. 0.32% (cross-target)  
- Same-Typology pairs: **1.08%** vs. 0.85%  

**Price tier is the most actionable dimension:**
- **(Low, Low):** highest co-purchase rate (2.72%) and highest total delayed revenue (€22.7k)  
- **(Medium, Medium):** highest expected value per exposure (€7.93) and €20.4k total revenue  
- **(High, High):** near-zero conversion (0.09%), confirming strong friction for high-ticket bundles  

A critical behavioural signal emerges: **81.3% of delayed co-purchases were added to cart together in the initial session**, confirming the cart as the decisive indicator of bundle intent and the key trigger for recovery strategies.

Timing dynamics reinforce this:
- Median delay from first co-view to co-purchase: **< 24 hours**  
- Conversion probability drops sharply after **48 hours**, indicating a narrow opportunity window for intervention.

### **7b. Business Recommendations**


**1. Prioritise the First-Session Cart Action as the Primary Micro-Conversion**

The single most consistent finding across driver tables, Kaplan–Meier curves, and the Cox model is that intent expressed early is the strongest predictor of conversion speed. The session-based survival analysis identifies a clear *“Golden Window”* concentrated in sessions 1–5, where purchase probability peaks (≈21.5% at sessions 4–5) before declining sharply; beyond session 10–12, conversion becomes marginal. This indicates that the objective of the first session should not be the purchase itself, but the activation of a high-intent signal — most notably the add-to-cart action.

Versace should therefore treat **add-to-cart in session 1 as the primary micro-conversion**, designing product pages to minimise friction and accelerate commitment. Practical interventions include one-tap mobile add-to-cart, real-time stock visibility, and persistent size selection. Given that the median time-to-purchase is under 48 hours for high-intent users, abandoned cart recovery should be triggered within hours rather than days, aligning intervention timing with the natural decay of purchase probability.

---

**2. Restructure Retargeting Around Behavioural Clusters, Not Channels**

The K-means segmentation and survival curves demonstrate that user behaviour is structurally heterogeneous, and a single retargeting strategy cannot effectively address all segments. Cluster C (HR = 2.48) consists of fast, impulse-driven buyers who respond to immediate follow-up within a narrow time window. Cluster B, characterised by repeated product views (avg. 3.31) and longer TTP (2.72 days), reflects a more deliberate evaluation process requiring sustained engagement. Cluster A (High-Value Power Shoppers, €183k total revenue) shows the most exploratory and selective behaviour, with lower conversion hazard due to extensive browsing of premium items without guaranteed purchase.

Versace should map CRM and paid audiences to these behavioural profiles and apply differentiated strategies in timing, messaging, and creative execution. Rapid, high-frequency retargeting is effective for Cluster C, while Cluster B requires a slower, trust-building sequence across multiple sessions. For Cluster A, short-term urgency tactics risk creating friction; instead, communication should emphasise brand value, craftsmanship, and aspiration, aligning with their premium orientation and longer decision cycles.

---

**3. Focus Bundle Recovery on (Low, Low) and (Medium, Medium) Pairs Within 48 Hours**

The co-purchase analysis reveals that delayed bundle conversion is highly concentrated and strongly conditioned on early cart behaviour. Of the 166 successful delayed co-purchases, 81.3% were already added to cart together in the first session, making the cart the most reliable signal of bundle intent. At the same time, effectiveness is highly dependent on price tier: (Low, Low) pairs generate the highest volume (2.72% conversion rate; €22.7k revenue), while (Medium, Medium) pairs maximise value per exposure (€7.93 expected value; €20.4k total). In contrast, (High, High) combinations show near-zero delayed conversion (0.09%), making post-session recovery inefficient.

Versace should therefore restrict bundle recovery efforts to **carted pairs within the first session**, focusing specifically on Low–Low and Medium–Medium combinations and activating campaigns within a 48-hour window, after which conversion likelihood drops sharply. High-price bundles should instead be addressed during the session itself through complementary product suggestions. A more effective creative approach is to move beyond standard abandoned cart messaging and instead present **“Complete the Vision”** communications, where previously co-selected items are shown together as a curated look, reinforcing and validating the user’s initial styling choice.